# 01. 따릉이 × 지하철 전처리 (API 수집 → 전처리 → Parquet)

서울시 API와 **대여이력 병합 CSV**를 읽어 전처리하고, 결과를 Parquet으로 저장한 뒤 마지막에 DataFrame으로 확인합니다.
그래프·상관계수는 `02_시각화_따릉이_지하철.ipynb`가 이 결과 파일을 읽어서 그립니다.

## 목차 — 위에서부터 이 순서대로 실행
| 순서 | 절 | 하는 일 (한 줄 요약) | 만드는 파일 (`data/parquet_bike_subway/`) |
|---|---|---|---|
| 0 | 준비 | 라이브러리 · 설정 · 공통 함수 · API 함수 | - |
| 1 | 달력 | 날짜별 요일·공휴일, 출퇴근 시간대 기준표 | `calendar_day`, `dim_hour` |
| 2 | 대여소 | API 2종 수집 → 두 출처 비교 → 좌표 오류 분리 | `bike_station_both`, `bike_station_diff`, `bike_station_mismatch` |
| 3 | 지하철역 ↔ 대여소 | 역 좌표 API → 환승역 묶기 → 300·500m 연결 | `subway_station_point`, `subway_station_phys`, `station_subway_link`, `bike_station_nearest` |
| 4 | 대여이력 | 병합 CSV(없으면 생성) → 정상/예외 분리 → 시간대 집계 | `mart_station_day_hour/`, `mart_station_month_hour/`, `rental_exception/` |
| 5 | 지하철 월별 승하차 | API → 시간대 세로(long) 변환 → 역 이름 매칭 | `subway_ride_month_hour`, `subway_name_map_month` |
| 6 | 서울교통공사 일별 승하차 | API → 정리 → 요일 프로필 → 일별 추정 | `subway_ride_day_hour`, `subway_daily_coverage`, `subway_day_profile_*`, `subway_ride_day_hour_full` |
| 7 | 통합 변수 | 대여소별 지하철 변수 | `feature_station_subway`, `feature_station_subway_month_hour` |
| 8 | 모델링 데이터 | 대여소 × 날짜 × 시간 + 변수 결합 | `model_station_day_hour/`, `model_feature_dictionary.csv` |
| 9 | 최종 확인 | 만든 Parquet을 DataFrame으로 불러와 확인 | - |

## 실행 전 준비
1. **인증키**: 환경변수 `SEOUL_API_KEY` 또는 Colab 보안 비밀(🔑) `SEOUL_API_KEY`. 둘 다 없으면 입력창이 뜹니다. 코드에 직접 쓰지 마세요.
2. **대여이력**: `따릉이_대여이력_병합` 노트북으로 만든 CSV가 있으면 `RENTAL_MERGED_CSV` 위치에 두세요(그대로 재사용). 없으면 원본 zip/csv를 `RENTAL_SOURCE_DIR`에 두면 같은 방식으로 자동 병합합니다.
3. **경로·기간**은 `0-2 설정` 셀에서만 바꿉니다.
4. 중간에 끊기면 **같은 셀을 다시 실행**하면 이어서 진행합니다 (API 응답·대여이력 청크를 저장해 두고 건너뜀).
5. 고장신고는 이 노트북 범위에서 제외했습니다.

# 0. 준비
## 0-1. 라이브러리
**왜:** 수집(`requests`)·전처리(`pandas`)·저장(`pyarrow`)·공휴일(`holidays`)에 필요한 패키지를 불러옵니다.

In [ ]:
# 필요한 패키지가 없을 때만 설치
import importlib, subprocess, sys

def ensure_package(name, pip_name=None):
    try:
        importlib.import_module(name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or name])

for _n in ["pandas", "numpy", "pyarrow", "requests", "holidays"]:
    ensure_package(_n)

import os, re, json, math, time, codecs, zipfile, getpass, datetime as dt
from pathlib import Path
from contextlib import contextmanager

import numpy as np
import pandas as pd
import requests
import holidays                      # 한국 공휴일 달력
import pyarrow.parquet as pq
from IPython.display import display

try:                                 # Colab이면 구글 드라이브 연결
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
print("pandas", pd.__version__, "/ Colab:", IN_COLAB)

## 0-2. 설정 ⭐ (여기만 바꾸면 됩니다)
**왜:** 기간·경로·인증키를 한 곳에서 정해 누가 실행해도 같은 결과가 나오게 합니다.

| 설정 | 뜻 |
|---|---|
| `START_DATE`, `END_DATE` | 분석 기간 (양 끝 포함). 병합 CSV 기간과 같게 |
| `PROJECT_DIR` | 프로젝트 폴더 (Colab: 드라이브 / 로컬: 환경변수 `BIKE_PROJECT_DIR`, 없으면 현재 폴더) |
| `RENTAL_SOURCE_DIR` | 대여이력 원본 zip/csv 폴더 (병합 CSV가 없을 때만 사용) |
| `RENTAL_MERGED_CSV` | 대여이력 병합 CSV (있으면 재사용) |
| `API_RAW_DIR` | API 응답 원본 저장 폴더 (다시 실행할 때 API를 또 호출하지 않음) |
| `OUT_DIR` | 전처리 결과 Parquet 폴더 → 02 시각화 노트북 입력 |
| `FORCE_API` | True면 저장된 API 응답을 무시하고 다시 받음 |

In [ ]:
# ▶ 분석 기간 (양 끝 포함)
START_DATE = "2024-01-01"
END_DATE   = "2026-06-30"

# ▶ 경로
if IN_COLAB:
    PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/final_project")
else:
    PROJECT_DIR = Path(os.environ.get("BIKE_PROJECT_DIR", ".")).resolve()
DATA_DIR          = PROJECT_DIR / "data"
RENTAL_SOURCE_DIR = DATA_DIR / "bike_raw"                                    # 대여이력 원본 zip/csv
RENTAL_MERGED_CSV = DATA_DIR / "bike_merged" / "대여이력_2024_2026상반기.csv"   # 병합 CSV (있으면 재사용)
API_RAW_DIR       = DATA_DIR / "api_raw"                                     # API 응답 원본
OUT_DIR           = DATA_DIR / "parquet_bike_subway"                         # 전처리 결과
TMP_DIR, LOG_DIR  = OUT_DIR / "_tmp", OUT_DIR / "_logs"                      # 중간 파일 / 진행 기록
for _d in (API_RAW_DIR, OUT_DIR, TMP_DIR, LOG_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# ▶ 서울 열린데이터광장 API (인증키: 환경변수 → Colab 보안 비밀 → 입력창)
BASE_URL = os.environ.get("SEOUL_API_BASE_URL", "http://openapi.seoul.go.kr:8088")
SEOUL_API_KEY = os.environ.get("SEOUL_API_KEY", "").strip()
if not SEOUL_API_KEY and IN_COLAB:
    try:
        from google.colab import userdata
        SEOUL_API_KEY = (userdata.get("SEOUL_API_KEY") or "").strip()
    except Exception:
        pass
if not SEOUL_API_KEY:
    SEOUL_API_KEY = getpass.getpass("서울 열린데이터광장 인증키: ").strip()
FORCE_API = False

# ▶ 기간 파생값
START_TS, END_TS = pd.Timestamp(START_DATE), pd.Timestamp(END_DATE)
END_EXCL      = END_TS + pd.Timedelta(days=1)                                # 기간 끝 다음날 0시
PERIOD_MONTHS = [p.strftime("%Y%m") for p in pd.period_range(START_DATE, END_DATE, freq="M")]
N_MONTHS      = len(PERIOD_MONTHS)
assert START_TS <= END_TS, "START_DATE가 END_DATE보다 늦습니다."

# ▶ 결과 폴더에 다른 기간 결과가 섞이지 않게 기록·확인
VERSION  = "bike_subway_v1"
manifest = LOG_DIR / "manifest.json"
settings = {"start_date": START_DATE, "end_date": END_DATE, "version": VERSION}
if manifest.exists():
    old = json.loads(manifest.read_text(encoding="utf-8"))
    if old != settings:
        raise RuntimeError(f"OUT_DIR에 다른 설정의 결과가 있습니다: {old}\n→ 폴더를 백업한 뒤 비우거나 OUT_DIR을 바꾸세요.")
else:
    manifest.write_text(json.dumps(settings, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"분석 기간 {START_DATE} ~ {END_DATE} ({N_MONTHS}개월)")
print("결과 폴더:", OUT_DIR)
print("병합 CSV:", RENTAL_MERGED_CSV, "(있음)" if RENTAL_MERGED_CSV.exists() else "(없음 → 4-1에서 생성)")

## 0-3. 공통 함수
**왜:** 결측 처리·형 변환·저장·거리 계산을 모든 절에서 같은 규칙으로 쓰기 위해서입니다.

| 함수 | 한 줄 요약 |
|---|---|
| `clean_text` | 앞뒤 공백 제거, `""`·`\N` → 결측 |
| `to_int`, `to_float` | 문자열 → 결측 허용 정수 / 실수 |
| `as_date` | 날짜 키 통일 (시각 제거) |
| `save_parquet` | 임시 파일에 쓰고 이름 변경 → 끊겨도 깨진 파일이 남지 않음 |
| `read_parquet_dir` | 폴더 안 Parquet 조각을 합쳐 읽기 |
| `haversine_m` | 두 좌표 사이 직선거리(m) |
| `normalize_name`, `normalize_line` | 역명·호선 표기 통일 |
| `swap_fix`, `valid_coord` | 위경도 뒤바뀜 교정, 서울권 좌표 확인 |

In [ ]:
MISSING_TOKENS = ["", "\\N"]          # 원본에서 결측을 뜻하는 값


def now_str():
    return dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def clean_text(s):
    t = s.astype("string").str.strip()
    return t.mask(t.isin(MISSING_TOKENS))


def to_float(s):
    return pd.to_numeric(clean_text(s), errors="coerce").astype("float64")


def to_int(s):
    x = to_float(s)
    return x.where(x.isna() | (x % 1 == 0)).astype("Int64")      # 소수는 정수로 바꾸지 않고 결측


def as_date(s):
    return pd.to_datetime(s).dt.normalize().astype("datetime64[ns]")


def smatch(s, pattern):
    return s.astype("string").str.match(pattern).fillna(False).astype(bool)


def save_parquet(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    df.to_parquet(tmp, index=False)
    os.replace(tmp, path)


def read_parquet_dir(folder, columns=None, pattern="*.parquet"):
    files = sorted(Path(folder).glob(pattern))
    return pd.concat([pd.read_parquet(f, columns=columns) for f in files], ignore_index=True) if files else pd.DataFrame()


def concat_nonempty(frames, **kw):
    frames = [f for f in frames if f is not None and len(f)]
    return pd.concat(frames, **kw) if frames else pd.DataFrame()


def read_csv_log(path, cols):
    return pd.read_csv(path, dtype={"key": str}) if Path(path).exists() else pd.DataFrame(columns=cols)


def append_csv_log(path, row):
    log = read_csv_log(path, list(row))
    log = pd.concat([log[log["key"] != row["key"]], pd.DataFrame([row])], ignore_index=True)
    log.to_csv(path, index=False, encoding="utf-8-sig")


# ---------------- 거리 · 역명 ----------------
NEAR_RADIUS_M     = 300       # 가까움 구간 경계
MAX_LINK_RADIUS_M = 500       # 연결 최대 반경
SAME_STATION_M    = 700       # 같은 이름의 역이 이 거리 이상 떨어지면 다른 역
MANUAL_NAME_MAP   = {}        # 역명 수동 매핑 예) {"총신대입구(이수)": "이수"}
SEOUL_LAT, SEOUL_LON = (37.0, 38.2), (126.0, 128.0)


def haversine_m(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371008.8 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def normalize_name(name):
    s = MANUAL_NAME_MAP.get(str(name).strip(), str(name).strip())
    s = re.sub(r"\(.*?\)", "", s)                                 # 괄호 내용 제거
    s = re.sub(r"\s+", "", s)
    return re.sub(r"역$", "", s)                                  # 끝의 '역' 제거


def normalize_line(line):
    s = str(line).strip()
    m = re.fullmatch(r"0*(\d+)\s*(호선)?", s)
    return f"{int(m.group(1))}호선" if m else re.sub(r"\s+", "", s)


def swap_fix(df, lat_col, lon_col):
    swapped = df[lat_col].between(124, 132) & df[lon_col].between(33, 39)
    df.loc[swapped, [lat_col, lon_col]] = df.loc[swapped, [lon_col, lat_col]].to_numpy()
    return int(swapped.sum())


def valid_coord(lat, lon):
    return lat.between(*SEOUL_LAT) & lon.between(*SEOUL_LON)

print("공통 함수 준비 완료")

## 0-4. 서울 열린데이터광장 API 함수
**왜:** 1회 최대 1,000행이라 페이지를 넘기며 끝까지 받고, 받은 응답은 `api_raw/`에 저장해 다시 실행할 때 API를 또 부르지 않습니다.

| 함수 | 한 줄 요약 |
|---|---|
| `iter_pages` | 1,000행씩 페이지를 차례로 받기 (중간 행부터 재개 가능) |
| `fetch_all` | 모든 페이지 합치기, 통신 실패만 3회 재시도 |
| `api_cached` | 저장된 응답이 있으면 읽고, 없으면 받아서 문자열 그대로 저장 |

- 응답 코드: `INFO-000` 정상 / `INFO-200` 자료 없음 / 그 밖(`INFO-100` 인증키 오류 등)은 즉시 중단
- 자료 없음 응답은 0건으로 단정하지 않고 저장하지 않습니다.

In [ ]:
PAGE_SIZE     = 1000
MAX_RETRIES   = 3
REQUEST_SLEEP = float(os.environ.get("SEOUL_API_SLEEP", 0.3))     # 페이지 사이 대기(초)


class ApiRequestError(RuntimeError):     # 통신 실패 → 재시도
    pass


class ApiResponseError(RuntimeError):    # 인증키 오류 등 API 오류 응답 → 중단
    pass


def _request_json(service, start, end, param=""):
    if SEOUL_API_KEY in ("", "sample"):
        raise ValueError("본인 서울 API 인증키가 필요합니다.")
    url = f"{BASE_URL}/{SEOUL_API_KEY}/json/{service}/{start}/{end}/" + (str(param) if param else "")
    try:
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        return r.json()
    except (requests.RequestException, ValueError):
        # 인증키가 들어 있는 URL이 오류 메시지에 노출되지 않게 from None
        raise ApiRequestError(f"{service}: {start}~{end}행 요청 실패 (조건값={param or '-'})") from None


def _parse(data, service):
    svc = data.get(service)
    if not isinstance(svc, dict):
        svc = next((v for v in data.values() if isinstance(v, dict) and ("row" in v or "RESULT" in v)), {})
    result = svc.get("RESULT") or data.get("RESULT") or {}
    return result.get("CODE"), result.get("MESSAGE"), svc.get("row") or [], svc.get("list_total_count")


def iter_pages(service, param="", start=1):
    """(시작행, 끝행, 페이지 DataFrame, 전체건수)를 차례로 돌려줌"""
    prev = None
    while True:
        end = start + PAGE_SIZE - 1
        code, msg, rows, total = _parse(_request_json(service, start, end, param), service)
        if code == "INFO-200":
            return
        if code not in (None, "INFO-000"):
            raise ApiResponseError(f"{service}: {code} / {msg}")
        if not rows:
            return
        if rows == prev:
            raise ApiResponseError(f"{service}: 이전 페이지와 같은 응답 (페이지 조건 무시됨)")
        yield start, end, pd.DataFrame(rows), total
        prev = rows
        if len(rows) < PAGE_SIZE:
            return
        try:
            if start + len(rows) - 1 >= int(total):
                return
        except (TypeError, ValueError):
            pass
        start += PAGE_SIZE
        time.sleep(REQUEST_SLEEP)


def fetch_all(service, param=""):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            frames = [df for _, _, df, _ in iter_pages(service, param)]
            return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
        except ApiRequestError:
            print(f"[요청 실패] {service} {param} {attempt}/{MAX_RETRIES}회")
            if attempt == MAX_RETRIES:
                raise
            time.sleep(attempt * 3)


def to_text_frame(df):
    # 원본 보관: 모든 값을 문자열(또는 결측)로 통일
    return pd.DataFrame({str(c): df[c].map(lambda v: None if pd.isna(v) else str(v)) for c in df.columns})


def api_cached(service, name, param="", force=None):
    path = API_RAW_DIR / f"{name}.parquet"
    if path.exists() and not (FORCE_API if force is None else force):
        return pd.read_parquet(path)
    df = fetch_all(service, param)
    if df.empty:                                  # 자료 없음은 저장하지 않음
        return df
    df = to_text_frame(df)
    df["_collected_at"] = now_str()
    save_parquet(df, path)
    return df

print("API 함수 준비 완료 / BASE_URL:", BASE_URL)

# 1. 달력 · 시간대 기준표
**왜:** 평일·주말·공휴일별 하루 평균을 내려면 기간 안 각 날짜의 요일 구분이 필요합니다.

| `calendar_day` 컬럼 | 뜻 |
|---|---|
| `cal_date` | 날짜 |
| `year`, `month`, `ym` | 연, 월, 연월(`YYYYMM`) |
| `dow` | 요일 (1=월 … 7=일) |
| `is_weekend`, `is_holiday`, `holiday_name` | 주말 여부, 공휴일 여부, 공휴일 이름 |
| `day_type` | `weekday` / `weekend`(토·일) / `holiday`(평일 공휴일) |

| `dim_hour` 컬럼 | 뜻 |
|---|---|
| `hour` | 0~23시 |
| `is_commute_morning`, `is_commute_evening` | 출근(7~9시), 퇴근(17~19시) |
| `hour_group` | 새벽 / 아침 / 출근 / 낮 / 퇴근 / 저녁 |

In [ ]:
EXTRA_HOLIDAYS = {}          # 빠진 임시공휴일 등은 공식 발표 확인 후 추가 예) {"2025-10-10": "임시공휴일"}

kr = holidays.KR(years=range(START_TS.year, END_TS.year + 1))
for _d, _n in EXTRA_HOLIDAYS.items():
    kr[pd.Timestamp(_d).date()] = _n

cal = pd.DataFrame({"cal_date": as_date(pd.Series(pd.date_range(START_DATE, END_DATE, freq="D")))})
cal["year"]         = cal["cal_date"].dt.year
cal["month"]        = cal["cal_date"].dt.month
cal["ym"]           = cal["cal_date"].dt.strftime("%Y%m").astype("string")
cal["dow"]          = cal["cal_date"].dt.dayofweek + 1
cal["is_weekend"]   = cal["dow"] >= 6
cal["is_holiday"]   = cal["cal_date"].dt.date.map(lambda d: d in kr).astype(bool)
cal["holiday_name"] = cal["cal_date"].dt.date.map(lambda d: kr.get(d)).astype("string")
cal["day_type"]     = pd.Series(np.where(cal["is_weekend"], "weekend", np.where(cal["is_holiday"], "holiday", "weekday")),
                                dtype="string")
save_parquet(cal, OUT_DIR / "calendar_day.parquet")

dim_hour = pd.DataFrame({"hour": np.arange(24, dtype="int64")})
dim_hour["is_commute_morning"] = dim_hour["hour"].between(7, 9)
dim_hour["is_commute_evening"] = dim_hour["hour"].between(17, 19)
dim_hour["hour_group"] = pd.cut(dim_hour["hour"], bins=[-1, 5, 6, 9, 16, 19, 23],
                                labels=["새벽", "아침", "출근", "낮", "퇴근", "저녁"]).astype("string")
save_parquet(dim_hour, OUT_DIR / "dim_hour.parquet")

DAYS_IN_PERIOD = len(cal)
DAYS_BY_TYPE   = cal["day_type"].value_counts()
display(cal.groupby(["year", "day_type"]).size().unstack(fill_value=0))
print("평일에 낀 공휴일 (빠진 날이 없는지 확인)")
display(cal.loc[cal["is_holiday"] & ~cal["is_weekend"], ["cal_date", "holiday_name"]].reset_index(drop=True))

# 2. 따릉이 대여소
## 2-1. API 수집 (대여소 정보 · 대여소 마스터)
**왜:** 대여소 ID·좌표는 대여이력·지하철역을 잇는 기준이므로 두 출처(따릉이 / 서울시 교통정보시스템)를 모두 받아 비교합니다.

| 서비스 | 출처 | 주요 원본 컬럼 |
|---|---|---|
| `tbCycleStationInfo` | 따릉이 대여소 정보 | `STA_LOC`, `RENT_ID`, `RENT_NO`, `RENT_NM`, `HOLD_NUM`, `STA_ADD1/2`, `STA_LAT`, `STA_LONG` |
| `bikeStationMaster` | 교통정보시스템 대여소 마스터 | `RNTLS_ID`, `ADDR1/2`, `LAT`, `LOT` |

In [ ]:
raw_station = api_cached("tbCycleStationInfo", "bike_station_info")
raw_master  = api_cached("bikeStationMaster",  "bike_station_master")
assert len(raw_station) and len(raw_master), "대여소 API 응답이 비었습니다. 인증키·서비스명을 확인하세요."
print("대여소 정보:", raw_station.shape, "/ 대여소 마스터:", raw_master.shape, "/ 수집 시각:", raw_station["_collected_at"].iloc[0])
display(raw_station.head(3))
display(raw_master.head(3))

## 2-2. 컬럼 정리 · 형 변환
**왜:** 원본은 전부 문자열이라 번호·거치대는 정수, 좌표는 실수로 바꾸고 주소 두 칸을 합칩니다.

| 컬럼 | 뜻 |
|---|---|
| `sta_loc` | 자치구 |
| `rent_id` | 대여소 ID (`ST-xxx`, 대여이력 `rental_station_id`와 같은 값) |
| `rent_no` | 대여소 번호 (`00108` → 108) |
| `rent_nm` | 대여소 이름 |
| `sta_addr` | 주소 (`주소1 , 주소2`, 주소2가 없으면 주소1) |
| `sta_lat`, `sta_long` | 위도, 경도 |
| `hold_num` | 거치대 수 |

In [ ]:
STATION_RENAME = {"STA_LOC": "sta_loc", "RENT_ID": "rent_id", "RENT_NO": "rent_no", "RENT_NM": "rent_nm",
                  "HOLD_NUM": "hold_num", "STA_ADD1": "sta_add1", "STA_ADD2": "sta_add2",
                  "STA_LAT": "sta_lat", "STA_LONG": "sta_long"}
MASTER_RENAME  = {"RNTLS_ID": "rntls_id", "ADDR1": "addr1", "ADDR2": "addr2", "LAT": "lat", "LOT": "lot"}


def join_addr(a1, a2):
    a1, a2 = a1.fillna(""), a2.fillna("")
    return a1.where(a2 == "", a1 + " , " + a2).astype("string")


# 따릉이 대여소 정보
df_station2 = raw_station.rename(columns=STATION_RENAME)[list(STATION_RENAME.values())].copy()
for c in ["sta_loc", "rent_id", "rent_nm", "sta_add1", "sta_add2"]:
    df_station2[c] = df_station2[c].astype("string").str.strip()
df_station2["rent_no"]  = to_int(df_station2["rent_no"])
df_station2["hold_num"] = to_int(df_station2["hold_num"])
df_station2["sta_lat"]  = to_float(df_station2["sta_lat"])
df_station2["sta_long"] = to_float(df_station2["sta_long"])
df_station2["sta_addr"] = join_addr(df_station2["sta_add1"], df_station2["sta_add2"])
df_station2 = df_station2[["sta_loc", "rent_id", "rent_no", "rent_nm", "sta_addr", "sta_lat", "sta_long", "hold_num"]]

# 교통정보시스템 대여소 마스터
df_master2 = raw_master.rename(columns=MASTER_RENAME)[list(MASTER_RENAME.values())].copy()
for c in ["rntls_id", "addr1", "addr2"]:
    df_master2[c] = df_master2[c].astype("string").str.strip()
df_master2["lat"]  = to_float(df_master2["lat"])
df_master2["lot"]  = to_float(df_master2["lot"])
df_master2["addr"] = join_addr(df_master2["addr1"], df_master2["addr2"])
df_master2 = df_master2[["rntls_id", "addr", "lat", "lot"]]

for name, d, id_col, lat, lon in [("대여소 정보", df_station2, "rent_id", "sta_lat", "sta_long"),
                                   ("대여소 마스터", df_master2, "rntls_id", "lat", "lot")]:
    bad = int(((d[lat] == 0) | (d[lon] == 0) | d[lat].isna() | d[lon].isna()).sum())
    print(f"[{name}] {len(d):,}행 / 좌표 0·결측 {bad}건 / ID 중복 {int(d[id_col].duplicated().sum())}건")
display(df_station2.head())

## 2-3. 두 출처 비교 (`df_both` / `df_diff` / `df_mismatch`)
**왜:** 양쪽에 같은 ID·좌표로 있는 대여소만 믿고 쓰고, 한쪽에만 있거나 값이 다른 대여소는 따로 보관합니다.

| 결과 | 뜻 |
|---|---|
| `df_both` | 양쪽에 있고 좌표가 같은 대여소 (주소만 다른 경우는 유지) |
| `df_diff` | 한쪽에만 있는 대여소 + `data_source`(`따릉이` / `교통정보시스템`) |
| `df_mismatch` | 주소·좌표가 다른 대여소 (`diff_cols`: 다른 항목, `lat_gap`·`long_gap`: 좌표 차이) |

In [ ]:
COORD_TOL = 1e-5                         # 좌표 허용 오차(도) ≈ 1.1m
keep_cols = ["sta_loc", "rent_id", "rent_no", "rent_nm", "sta_addr", "sta_lat", "sta_long", "hold_num"]

m = df_master2.rename(columns={"rntls_id": "rent_id", "addr": "sta_addr", "lat": "sta_lat", "lot": "sta_long"})
ids_bike, ids_master = set(df_station2["rent_id"].dropna()), set(m["rent_id"].dropna())

# 1) 교집합 (접미사 _master = 교통정보시스템 값), 1:1 결합인지 검증
df_both = df_station2.merge(
    m.rename(columns={"sta_addr": "sta_addr_master", "sta_lat": "sta_lat_master", "sta_long": "sta_long_master"}),
    on="rent_id", how="inner", validate="one_to_one")

# 2) 한쪽에만 있는 대여소
bike_only   = df_station2[~df_station2["rent_id"].isin(ids_master)].assign(data_source="따릉이")
master_only = m[~m["rent_id"].isin(ids_bike)].assign(data_source="교통정보시스템")


def fix_station_dtypes(df):
    # 합친 뒤 자료형 고정
    df = df.reindex(columns=keep_cols + (["data_source"] if "data_source" in df.columns else []))
    for c in ["sta_loc", "rent_id", "rent_nm", "sta_addr", "data_source"]:
        if c in df.columns:
            df[c] = df[c].astype("string")
    for c in ["rent_no", "hold_num"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
    for c in ["sta_lat", "sta_long"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("float64")
    return df


df_diff = concat_nonempty([bike_only, master_only], ignore_index=True)
df_diff = fix_station_dtypes(df_diff if len(df_diff) else pd.DataFrame(columns=keep_cols + ["data_source"]))

# 3) 주소·위도·경도 비교
def norm_addr(s):
    return s.fillna("").str.replace(r"\s+", " ", regex=True).str.strip()

def same_coord(a, b):
    a, b = a.to_numpy(dtype="float64", na_value=np.nan), b.to_numpy(dtype="float64", na_value=np.nan)
    return np.isclose(a, b, rtol=0, atol=COORD_TOL) | (np.isnan(a) & np.isnan(b))

same_addr = (norm_addr(df_both["sta_addr"]) == norm_addr(df_both["sta_addr_master"])).to_numpy(bool)
same_lat  = same_coord(df_both["sta_lat"], df_both["sta_lat_master"])
same_long = same_coord(df_both["sta_long"], df_both["sta_long_master"])
all_same, same_xy = same_addr & same_lat & same_long, same_lat & same_long

mismatch_path = OUT_DIR / "bike_station_mismatch.parquet"
if (~all_same).any():
    df_mismatch = df_both.loc[~all_same].copy()
    flags = np.column_stack([~same_addr, ~same_lat, ~same_long])[~all_same]
    names = np.array(["주소", "위도", "경도"])
    df_mismatch["diff_cols"] = pd.Series([",".join(names[r]) for r in flags], index=df_mismatch.index, dtype="string")
    df_mismatch["lat_gap"]  = (df_mismatch["sta_lat"] - df_mismatch["sta_lat_master"]).abs()
    df_mismatch["long_gap"] = (df_mismatch["sta_long"] - df_mismatch["sta_long_master"]).abs()
    df_mismatch = df_mismatch.reset_index(drop=True)
    save_parquet(df_mismatch, mismatch_path)
    print("불일치 항목별 건수:\n", df_mismatch["diff_cols"].value_counts().to_string())
else:
    df_mismatch = pd.DataFrame()
    mismatch_path.unlink(missing_ok=True)      # 이전 실행의 파일이 남지 않게

# 4) 좌표가 다른 대여소만 거리 분석에서 제외 (주소만 다른 경우는 유지)
df_both = df_both.loc[same_xy, keep_cols].reset_index(drop=True)
print(f"양쪽 모두 {len(df_both):,}곳 / 따릉이에만 {len(bike_only):,}곳 / 교통정보시스템에만 {len(master_only):,}곳 / 불일치 {len(df_mismatch):,}곳")

## 2-4. 좌표 오류 대여소 → `df_diff`에 추가
**왜:** 양쪽에 있어도 좌표가 0이거나 서울권 밖이면 거리 계산이 불가능하므로 `data_source="따릉이,교통정보시스템"`으로 따로 표시합니다 (`df_both`에서 지우지는 않음).

In [ ]:
BOTH_SOURCE = "따릉이,교통정보시스템"       # 두 곳에 있으나 좌표 오류로 거리 계산 불가

chk = df_both.copy()
n_swapped_bike = swap_fix(chk, "sta_lat", "sta_long")
err_both = chk[~valid_coord(chk["sta_lat"], chk["sta_long"])]
df_err_add = err_both[keep_cols].assign(data_source=BOTH_SOURCE)
mem_new = df_err_add[~df_err_add["rent_id"].isin(df_diff["rent_id"])]
df_diff = fix_station_dtypes(concat_nonempty([df_diff, mem_new], ignore_index=True))
print(f"좌표 오류 {len(err_both)}곳 (위경도 뒤바뀜 교정 {n_swapped_bike}곳) → df_diff에 추가 {len(mem_new)}곳")
display(df_diff["data_source"].value_counts().to_frame("대여소 수"))

## 2-5. 저장
| 파일 | 한 행 | 키 |
|---|---|---|
| `bike_station_both.parquet` | 양쪽에 있고 좌표가 같은 대여소 | `rent_id` |
| `bike_station_diff.parquet` | 한쪽에만 있거나 좌표 오류인 대여소 (+ `data_source`) | `rent_id` |
| `bike_station_mismatch.parquet` | 주소·좌표가 다른 대여소 (있을 때만) | `rent_id` |

In [ ]:
assert df_both["rent_id"].is_unique and df_diff["rent_id"].is_unique, "대여소 ID 중복"
save_parquet(df_both, OUT_DIR / "bike_station_both.parquet")
save_parquet(df_diff, OUT_DIR / "bike_station_diff.parquet")
print("저장:", len(df_both), "/", len(df_diff), "행")

# 3. 지하철역 ↔ 대여소 근접
## 3-1. 지하철역 API → 물리역(환승역 묶기)
**왜:** 같은 역이 호선별로 여러 행이라, 같은 위치의 역을 하나(물리역 `phys_key`)로 묶어야 대여소와 1:1로 거리를 잴 수 있습니다. 지도 API 없이 좌표로 직접 계산합니다.

| 컬럼 | 뜻 |
|---|---|
| `bldn_id`, `bldn_nm`, `route` | 역 ID, 역명, 호선 |
| `lat`, `lot` | 위도, 경도 |
| `name_key`, `line_key` | 정규화한 역명 / 호선 (`01호선` → `1호선`) |
| `phys_key` | 물리역 키. 같은 이름인데 700m 넘게 떨어지면 `역명_호선`으로 분리 |
| `subway_name`, `lines`, `n_lines` | (물리역 표) 역명, 지나는 호선, 호선 수 |

In [ ]:
raw_subway = api_cached("subwayStationMaster", "subway_station_master")
assert len(raw_subway), "지하철역 API 응답이 비었습니다."
df_subway2 = raw_subway.rename(columns={"BLDN_ID": "bldn_id", "BLDN_NM": "bldn_nm", "ROUTE": "route",
                                        "LAT": "lat", "LOT": "lot"})[["bldn_id", "bldn_nm", "route", "lat", "lot"]].copy()
for c in ["bldn_id", "bldn_nm", "route"]:
    df_subway2[c] = df_subway2[c].astype("string").str.strip()
df_subway2["lat"], df_subway2["lot"] = to_float(df_subway2["lat"]), to_float(df_subway2["lot"])

n_swapped = swap_fix(df_subway2, "lat", "lot")
ok = valid_coord(df_subway2["lat"], df_subway2["lot"])
df_subway_err = df_subway2[~ok]
df_subway_pt = df_subway2[ok].drop_duplicates("bldn_id").reset_index(drop=True)
df_subway_pt["name_key"] = df_subway_pt["bldn_nm"].map(normalize_name).astype("string")
df_subway_pt["line_key"] = df_subway_pt["route"].map(normalize_line).astype("string")

# 같은 이름 = 같은 위치. 단, 700m 넘게 퍼져 있으면 호선별로 분리
df_subway_pt["phys_key"] = df_subway_pt["name_key"]
for name_key, g in df_subway_pt.groupby("name_key"):
    if haversine_m(g["lat"], g["lot"], g["lat"].mean(), g["lot"].mean()).max() > SAME_STATION_M:
        df_subway_pt.loc[g.index, "phys_key"] = name_key + "_" + g["line_key"]
df_subway_pt["phys_key"] = df_subway_pt["phys_key"].astype("string")

df_subway_station = (df_subway_pt.groupby("phys_key")
                     .agg(subway_name=("bldn_nm", "first"), lat=("lat", "mean"), lot=("lot", "mean"),
                          lines=("line_key", lambda s: ", ".join(sorted(s.unique()))), n_lines=("line_key", "nunique"))
                     .reset_index())
save_parquet(df_subway_pt, OUT_DIR / "subway_station_point.parquet")
save_parquet(df_subway_station, OUT_DIR / "subway_station_phys.parquet")
print(f"원본 {len(df_subway2):,}행 → 유효 {len(df_subway_pt):,}행 / 서울권 밖 {len(df_subway_err)}행 / 위경도 교정 {n_swapped}행")
print(f"물리역 {len(df_subway_station):,}곳 / 환승역 {(df_subway_station['n_lines'] >= 2).sum():,}곳")
display(df_subway_station.head())

## 3-2. 거리 계산 대상 대여소
**왜:** 운영 중인 따릉이 대여소만 연결합니다. `bike_station_both` 전부 + `bike_station_diff` 중 `따릉이`(따릉이에만 있음)를 쓰고, 교통정보시스템에만 있는 대여소와 좌표 오류 대여소는 뺍니다.

| 컬럼 | 뜻 |
|---|---|
| `station_source` | `양쪽일치` / `따릉이만` |

In [ ]:
df_bike_st = concat_nonempty([
    df_both[["rent_id", "rent_no", "rent_nm", "sta_lat", "sta_long"]].assign(station_source="양쪽일치"),
    df_diff.loc[df_diff["data_source"] == "따릉이", ["rent_id", "rent_no", "rent_nm", "sta_lat", "sta_long"]].assign(station_source="따릉이만"),
], ignore_index=True)
df_bike_st["station_source"] = df_bike_st["station_source"].astype("string")
n_swapped = swap_fix(df_bike_st, "sta_lat", "sta_long")
ok = valid_coord(df_bike_st["sta_lat"], df_bike_st["sta_long"])
df_bike_st_err, df_bike_st = df_bike_st[~ok], df_bike_st[ok].reset_index(drop=True)
assert df_bike_st["rent_id"].is_unique
print(f"사용 {len(df_bike_st):,}곳 / 좌표 오류 제외 {len(df_bike_st_err)}곳 / 위경도 교정 {n_swapped}곳")
display(df_bike_st["station_source"].value_counts().to_frame("대여소 수"))

## 3-3. 대여소-지하철역 연결 (`station_subway_link`)
**왜:** 대여소마다 500m 안의 역을 모두 찾고 가장 가까운 역을 표시해, 이후 승하차 자료와 `phys_key`로 연결합니다.

| `station_subway_link` 컬럼 | 뜻 |
|---|---|
| `rent_id`, `rent_no`, `rent_nm` | 대여소 ID, 번호, 이름 |
| `station_source` | `양쪽일치` / `따릉이만` |
| `phys_key`, `subway_name`, `lines` | 물리역 키, 역명, 호선 |
| `distance_m` | 대여소-역 직선거리(m) |
| `radius_band` | `0-300m` / `300-500m` |
| `subway_rank`, `is_nearest` | 가까운 역 순서 (1=가장 가까움), 가장 가까운 역 여부 |
| `radius_m` | 연결 최대 반경(500) |

| `bike_station_nearest` 컬럼 | 뜻 |
|---|---|
| `rent_id`, `station_source` | 대여소 ID, 출처 |
| `nearest_any_phys_key`, `nearest_any_subway_name` | 거리 제한 없이 가장 가까운 역 |
| `nearest_any_m` | 그 역까지 거리(m, 500m 넘어도 기록) |

In [ ]:
dist = haversine_m(df_bike_st["sta_lat"].to_numpy()[:, None], df_bike_st["sta_long"].to_numpy()[:, None],
                   df_subway_station["lat"].to_numpy()[None, :], df_subway_station["lot"].to_numpy()[None, :])  # (대여소, 역)

bi, si = np.where(dist <= MAX_LINK_RADIUS_M)
df_link = pd.DataFrame({
    "rent_id":        df_bike_st["rent_id"].to_numpy()[bi],
    "rent_no":        df_bike_st["rent_no"].to_numpy()[bi],
    "rent_nm":        df_bike_st["rent_nm"].to_numpy()[bi],
    "station_source": df_bike_st["station_source"].to_numpy()[bi],
    "phys_key":       df_subway_station["phys_key"].to_numpy()[si],
    "subway_name":    df_subway_station["subway_name"].to_numpy()[si],
    "lines":          df_subway_station["lines"].to_numpy()[si],
    "distance_m":     dist[bi, si].round(1),
})
for c in ["rent_id", "rent_nm", "station_source", "phys_key", "subway_name", "lines"]:
    df_link[c] = df_link[c].astype("string")
df_link["rent_no"]     = pd.array(df_link["rent_no"], dtype="Int64")
df_link["radius_band"] = pd.Series(np.where(df_link["distance_m"] <= NEAR_RADIUS_M, f"0-{NEAR_RADIUS_M}m",
                                            f"{NEAR_RADIUS_M}-{MAX_LINK_RADIUS_M}m"), dtype="string")
df_link["subway_rank"] = df_link.groupby("rent_id")["distance_m"].rank(method="first").astype("int64")
df_link["is_nearest"]  = df_link["subway_rank"].eq(1)
df_link["radius_m"]    = MAX_LINK_RADIUS_M
df_link = df_link.sort_values(["rent_id", "subway_rank"]).reset_index(drop=True)
assert not df_link.duplicated(["rent_id", "phys_key"]).any()
assert df_link.groupby("rent_id")["is_nearest"].sum().eq(1).all()
save_parquet(df_link, OUT_DIR / "station_subway_link.parquet")

# 반경 밖까지 포함한 가장 가까운 역 (시각화 노트북의 거리 분포용)
j = dist.argmin(axis=1)
bike_nearest = pd.DataFrame({"rent_id": df_bike_st["rent_id"].astype("string"),
                             "station_source": df_bike_st["station_source"],
                             "nearest_any_phys_key": pd.array(df_subway_station["phys_key"].to_numpy()[j], dtype="string"),
                             "nearest_any_subway_name": pd.array(df_subway_station["subway_name"].to_numpy()[j], dtype="string"),
                             "nearest_any_m": dist.min(axis=1).round(1)})
save_parquet(bike_nearest, OUT_DIR / "bike_station_nearest.parquet")

for r in (NEAR_RADIUS_M, MAX_LINK_RADIUS_M):
    print(f"가장 가까운 역이 {r}m 이내인 대여소: {(bike_nearest['nearest_any_m'] <= r).mean():.1%}")
print(f"연결 쌍 {len(df_link):,}건 / 역 근처 대여소 {df_link['rent_id'].nunique():,}곳 / 근처 대여소가 있는 역 {df_link['phys_key'].nunique():,}곳")
display(df_link.head())

# 4. 따릉이 대여이력
## 4-1. 병합 CSV 준비 (있으면 재사용, 없으면 생성)
**왜:** 월별 원본 파일(2024·2025 zip, 2026 csv)을 9개 컬럼만 골라 CSV 한 개로 합칩니다. `따릉이_대여이력_병합` 노트북과 같은 방식이며, 이미 만들어 둔 파일이 있으면 그대로 씁니다.

- 원본 파일명 끝의 `_YYMM.csv`로 월을 읽어 **분석 기간의 월만** 합칩니다 (빠진 월·중복 월이 있으면 중단).
- 쓰는 동안 끊겨도 반쯤 쓴 파일이 남지 않게 임시 파일에 쓴 뒤 이름을 바꿉니다.

| 병합 CSV 컬럼 | 뜻 |
|---|---|
| 자전거번호 | 자전거 ID |
| 대여일시, 반납일시 | `YYYY-MM-DD HH:MM:SS` |
| 대여 대여소번호, 대여 대여소명 | 대여 대여소 번호·이름 |
| 반납대여소번호, 반납대여소명 | 반납 대여소 번호·이름 |
| 대여대여소ID, 반납대여소ID | 대여소 ID (`ST-xxx`) — 대여소 표 `rent_id`와 같은 값 |

In [ ]:
RENTAL_USECOLS = ["자전거번호", "대여일시", "대여 대여소번호", "대여 대여소명", "반납일시",
                  "반납대여소번호", "반납대여소명", "대여대여소ID", "반납대여소ID"]
MERGE_CHUNK = 200_000
ENCODINGS   = ["utf-8-sig", "cp949", "euc-kr"]
NAME_RE     = re.compile(r"_(\d{2})(\d{2})\.csv$", re.IGNORECASE)     # '..._2401.csv' → 2024년 1월


@contextmanager
def open_member(path, inner=None):
    # 일반 파일이면 그대로, zip이면 내부 파일 열기
    if inner is None:
        with open(path, "rb") as f:
            yield f
    else:
        with zipfile.ZipFile(path) as z, z.open(inner) as f:
            yield f


def detect_encoding(path, inner=None):
    with open_member(path, inner) as f:
        sample = f.read(1 << 20)
    for enc in ENCODINGS:
        try:
            codecs.getincrementaldecoder(enc)().decode(sample, final=False)
            return enc
        except UnicodeDecodeError:
            continue
    raise ValueError(f"인코딩 판별 실패: {inner or path}")


def display_name(name):
    try:
        return name.encode("cp437").decode("cp949")     # zip 안 한글 파일명이 깨져 보일 때
    except (UnicodeEncodeError, UnicodeDecodeError):
        return name


def list_rental_sources(folder):
    """zip 내부 csv + 낱개 csv → [(연월, 표시이름, 경로, zip 내부경로)]"""
    out = []
    for p in sorted(Path(folder).rglob("*")):
        if p.suffix.lower() == ".zip":
            with zipfile.ZipFile(p) as z:
                for inner in z.namelist():
                    mm = NAME_RE.search(inner)
                    if mm:
                        out.append((f"20{mm.group(1)}{mm.group(2)}", f"{p.name}::{display_name(inner)}", p, inner))
        elif p.suffix.lower() == ".csv":
            mm = NAME_RE.search(p.name)
            if mm:
                out.append((f"20{mm.group(1)}{mm.group(2)}", p.name, p, None))
    return sorted(out, key=lambda x: x[0])


if RENTAL_MERGED_CSV.exists():
    print(f"병합 CSV 재사용: {RENTAL_MERGED_CSV} ({RENTAL_MERGED_CSV.stat().st_size / 1e9:.2f}GB)")
else:
    assert RENTAL_SOURCE_DIR.is_dir(), f"병합 CSV도 원본 폴더도 없습니다: {RENTAL_SOURCE_DIR}"
    sources = [s for s in list_rental_sources(RENTAL_SOURCE_DIR) if s[0] in PERIOD_MONTHS]
    found = [s[0] for s in sources]
    dup_m = sorted({x for x in found if found.count(x) > 1})
    miss_m = [x for x in PERIOD_MONTHS if x not in found]
    if dup_m or miss_m:
        raise RuntimeError(f"원본 월 확인 필요 — 중복 월: {dup_m} / 빠진 월: {miss_m} (빈 월을 0건으로 가정하지 않음)")

    RENTAL_MERGED_CSV.parent.mkdir(parents=True, exist_ok=True)
    tmp_csv = RENTAL_MERGED_CSV.with_name(RENTAL_MERGED_CSV.name + ".tmp")
    first, merge_rows = True, []
    for ym, label, path, inner in sources:
        enc, n = detect_encoding(path, inner), 0
        with open_member(path, inner) as f:
            for chunk in pd.read_csv(f, encoding=enc, usecols=RENTAL_USECOLS, dtype=str,
                                     keep_default_na=False, chunksize=MERGE_CHUNK):
                chunk[RENTAL_USECOLS].to_csv(tmp_csv, mode="w" if first else "a", header=first, index=False,
                                             encoding="utf-8-sig" if first else "utf-8")
                first, n = False, n + len(chunk)
        merge_rows.append({"ym": ym, "source": label, "rows": n})
        print(f"병합: {label} → {n:,}행")
    os.replace(tmp_csv, RENTAL_MERGED_CSV)
    pd.DataFrame(merge_rows).to_csv(LOG_DIR / "rental_merge_log.csv", index=False, encoding="utf-8-sig")
    print(f"병합 완료: {sum(r['rows'] for r in merge_rows):,}행 → {RENTAL_MERGED_CSV}")

## 4-2. 병합 CSV 확인
**왜:** 필요한 9개 컬럼이 모두 있는지 앞 5행으로 먼저 확인합니다.

In [ ]:
check = pd.read_csv(RENTAL_MERGED_CSV, nrows=5, dtype=str, encoding="utf-8-sig")
missing_cols = [c for c in RENTAL_USECOLS if c not in check.columns]
assert not missing_cols, f"병합 CSV에 없는 컬럼: {missing_cols}"
display(check)

## 4-3. 정상 / 예외 분리 + 청크별 부분 집계
**왜:** 약 1억 행이라 50만 행씩 읽으며 바로 대여소×날짜×시간 건수로 줄여 저장합니다. 끊겨도 다시 실행하면 완료된 청크는 건너뜁니다.

| 판정 | 처리 |
|---|---|
| 완전 중복 행 (같은 청크·바로 앞 청크) | 두 번째부터 제외 `DUPLICATE_ROW` |
| 대여일시 오류 / 기간 밖 | 제외 `RENTAL_DATE_BAD` / `OUT_OF_PERIOD` |
| 대여 대여소 ID가 `ST-숫자` 아님 | 제외 `RENTAL_STATION_BAD` |
| 위 조건 통과 | **대여 이벤트** (`rent_count`, 반납 누락도 포함) |
| 대여 이벤트 + 반납일시·반납 대여소 정상 + 반납 ≥ 대여 | **정상 완료** (`completed_rent_count`, 반납시각 기준 `return_count`) |
| 대여 이벤트지만 반납 이상 | 반납 문제 사유만 기록 (`RETURN_NULL` 등) |

- 제외된 행은 `rental_exception/`에 사유(`reason`)와 함께 원본 그대로 저장합니다.

In [ ]:
RENTAL_RENAME = {"자전거번호": "bike_no", "대여일시": "rental_date", "대여 대여소번호": "rental_station_no",
                 "대여 대여소명": "rental_station_nm", "반납일시": "return_date", "반납대여소번호": "return_station_no",
                 "반납대여소명": "return_station_nm", "대여대여소ID": "rental_station_id", "반납대여소ID": "return_station_id"}
RCOLS          = list(RENTAL_RENAME.values())
CHUNK_ROWS     = 500_000
DT_FMT         = "%Y-%m-%d %H:%M:%S"
REASONS        = ["DUPLICATE_ROW", "RENTAL_DATE_BAD", "OUT_OF_PERIOD", "RENTAL_STATION_BAD"]
RETURN_REASONS = ["RETURN_NULL", "RETURN_BAD_FORMAT", "RETURN_BEFORE_RENTAL", "RETURN_STATION_BAD"]
PART_DIR       = TMP_DIR / "rental_parts"
EXC_DIR        = OUT_DIR / "rental_exception"
SPLIT_LOG      = LOG_DIR / "rental_split_log.csv"
_st            = RENTAL_MERGED_CSV.stat()
MERGED_SIG     = f"{_st.st_size}_{int(_st.st_mtime)}"           # 병합 CSV가 바뀌면 이어하기 금지


def agg_count(station, ts, mask, name):
    d = pd.DataFrame({"station_id": station[mask].astype("string"), "d": ts[mask].dt.normalize(),
                      "hour": ts[mask].dt.hour.astype("int64")})
    return d.groupby(["station_id", "d", "hour"]).size().rename(name).reset_index()


def split_chunk(df, prev_hash):
    """원본 청크 → (대여 집계, 완료 집계, 반납 집계, 예외 행, 건수, 행 해시)"""
    h = pd.util.hash_pandas_object(df[RCOLS], index=False).to_numpy()
    dup = pd.Series(pd.Series(h).duplicated().to_numpy() | np.isin(h, prev_hash), index=df.index)
    rd_txt, td_txt = clean_text(df["rental_date"]), clean_text(df["return_date"])
    rd = pd.to_datetime(rd_txt, format=DT_FMT, errors="coerce")
    td = pd.to_datetime(td_txt, format=DT_FMT, errors="coerce")
    rst, tst = clean_text(df["rental_station_id"]), clean_text(df["return_station_id"])
    rst_ok, tst_ok = smatch(rst, r"^ST-[0-9]+$"), smatch(tst, r"^ST-[0-9]+$")

    reject = pd.Series(pd.NA, index=df.index, dtype="string")      # 첫 번째로 걸린 사유
    reject[rd.isna()] = "RENTAL_DATE_BAD"
    reject[reject.isna() & ((rd < START_TS) | (rd >= END_EXCL))] = "OUT_OF_PERIOD"
    reject[reject.isna() & ~rst_ok] = "RENTAL_STATION_BAD"
    reject[dup] = "DUPLICATE_ROW"

    valid = reject.isna()
    completed = valid & td.notna() & (td >= rd) & tst_ok
    issue = pd.Series(pd.NA, index=df.index, dtype="string")
    issue[valid & td_txt.isna()] = "RETURN_NULL"
    issue[valid & td_txt.notna() & td.isna()] = "RETURN_BAD_FORMAT"
    issue[valid & td.notna() & (td < rd)] = "RETURN_BEFORE_RENTAL"
    issue[valid & td.notna() & (td >= rd) & ~tst_ok] = "RETURN_STATION_BAD"
    assert int(valid.sum()) == int(completed.sum()) + int((valid & issue.notna()).sum())

    counts = {"raw": len(df), "event": int(valid.sum()), "main": int(completed.sum()),
              "return_issue": int((valid & ~completed).sum()),
              **{r: int((reject == r).sum()) for r in REASONS},
              **{r: int((issue == r).sum()) for r in RETURN_REASONS}}
    exc = df[~valid].assign(reason=reject[~valid])
    return (agg_count(rst, rd, valid, "rent_count"), agg_count(rst, rd, completed, "completed_rent_count"),
            agg_count(tst, td, completed, "return_count"), exc, counts, h)


log = read_csv_log(SPLIT_LOG, ["key", "sig"])
if len(log) and (log["sig"].astype(str) != MERGED_SIG).any():
    raise RuntimeError("병합 CSV가 바뀌었습니다. OUT_DIR/_tmp/rental_parts, OUT_DIR/rental_exception, "
                       "_logs/rental_split_log.csv 를 지우고 다시 실행하세요.")
done = set(log["key"])
if "__END__" in done:
    print("대여이력 분리·부분집계: 이미 완료 → 건너뜀")
else:
    t0, prev_hash, n_chunks = time.time(), np.array([], dtype="uint64"), 0
    reader = pd.read_csv(RENTAL_MERGED_CSV, dtype=str, keep_default_na=False, encoding="utf-8-sig", chunksize=CHUNK_ROWS)
    for i, raw in enumerate(reader):
        key, n_chunks = f"c{i:05d}", i + 1
        df = raw.rename(columns=RENTAL_RENAME)[RCOLS]
        if key in done:                                        # 완료된 청크: 중복 비교용 해시만 계산
            prev_hash = pd.util.hash_pandas_object(df, index=False).to_numpy()
            continue
        rent, comp, ret, exc, counts, prev_hash = split_chunk(df, prev_hash)
        for kind, part in (("rent", rent), ("completed", comp), ("return", ret)):
            if len(part):
                save_parquet(part, PART_DIR / kind / f"{key}.parquet")
        if len(exc):
            save_parquet(exc, EXC_DIR / f"{key}.parquet")
        append_csv_log(SPLIT_LOG, {"key": key, "sig": MERGED_SIG, **counts})
        if (i + 1) % 20 == 0:
            print(f"  청크 {i + 1}개 처리 ({time.time() - t0:,.0f}초)")
    append_csv_log(SPLIT_LOG, {"key": "__END__", "sig": MERGED_SIG, "raw": n_chunks})
    print(f"완료: 청크 {n_chunks}개 ({time.time() - t0:,.0f}초)")

## 4-4. 월별 집계표 (`mart_station_day_hour/`, `mart_station_month_hour/`)
**왜:** 대여소 × 날짜 × 시간(일별)과 대여소 × 연월 × 요일구분 × 시간(월별) 두 표로 만들어 EDA·모델링·지하철 연결에 씁니다.

| 컬럼 | 뜻 |
|---|---|
| `station_id` | 대여소 ID (`ST-xxx`) |
| `d` | 날짜 (일별 표) |
| `ym`, `day_type`, `n_days` | 연월, 요일구분, 그 달의 해당 요일구분 일수 (월별 표) |
| `hour` | 0~23시 |
| `rent_count` | 대여 이벤트 건수 (대여시각 기준) |
| `completed_rent_count` | 정상 완료 이용 건수 (대여시각 기준) |
| `return_count` | 정상 반납 건수 (반납시각 기준, 기간 밖 반납 제외) |

In [ ]:
MART_B_DIR, MART_A_DIR = OUT_DIR / "mart_station_day_hour", OUT_DIR / "mart_station_month_hour"
MART_LOG   = LOG_DIR / "mart_build_log.csv"
PART_FILES = {k: [str(f) for f in sorted((PART_DIR / k).glob("*.parquet"))] for k in ("rent", "completed", "return")}
assert PART_FILES["rent"], "4-3 부분 집계가 없습니다."
VALUE_COL  = {"rent": "rent_count", "completed": "completed_rent_count", "return": "return_count"}
KEYS       = ["station_id", "d", "hour"]


def read_part(kind, d_from, d_to):
    """[d_from, d_to) 날짜의 부분 집계를 읽어 키별로 합침"""
    if not PART_FILES[kind]:
        return pd.DataFrame(columns=KEYS + [VALUE_COL[kind]])
    t = pq.read_table(PART_FILES[kind], filters=[("d", ">=", pd.Timestamp(d_from)), ("d", "<", pd.Timestamp(d_to))])
    df = t.to_pandas()
    if df.empty:
        return pd.DataFrame(columns=KEYS + [VALUE_COL[kind]])
    df["d"] = as_date(df["d"])
    return df.groupby(KEYS, as_index=False)[VALUE_COL[kind]].sum()


def build_month(ym):
    m0 = pd.Timestamp(f"{ym[:4]}-{ym[4:]}-01")
    m1 = m0 + pd.offsets.MonthBegin(1)
    parts = [read_part(k, m0, m1) for k in ("rent", "completed", "return")]
    for p in parts:
        p["d"] = as_date(p["d"]) if len(p) else p["d"].astype("datetime64[ns]")
        p["hour"] = p["hour"].astype("int64")
        p["station_id"] = p["station_id"].astype("string")
    b = (parts[0].merge(parts[1], on=KEYS, how="outer", validate="one_to_one")
                 .merge(parts[2], on=KEYS, how="outer", validate="one_to_one"))
    for c in ("rent_count", "completed_rent_count", "return_count"):
        b[c] = b[c].fillna(0).astype("int64")
    assert (b["completed_rent_count"] <= b["rent_count"]).all(), "완료 건수가 대여 건수보다 큽니다."
    b["ym"] = pd.Series(ym, index=b.index, dtype="string")
    b = b.sort_values(KEYS).reset_index(drop=True)
    save_parquet(b, MART_B_DIR / f"mart_station_day_hour_{ym}.parquet")

    a = (b.merge(cal[["cal_date", "day_type"]], left_on="d", right_on="cal_date", how="left", validate="many_to_one")
          .groupby(["station_id", "ym", "day_type", "hour"], as_index=False)[["rent_count", "completed_rent_count", "return_count"]].sum())
    a["n_days"] = a["day_type"].map(cal[cal["ym"] == ym].groupby("day_type").size()).astype("int64")
    save_parquet(a, MART_A_DIR / f"mart_station_month_hour_{ym}.parquet")
    append_csv_log(MART_LOG, {"key": ym, "b_rows": len(b), "a_rows": len(a)})
    return len(b), len(a)


late_return = int(read_part("return", END_EXCL, "2100-01-01")["return_count"].sum())   # 기간 끝 이후 반납 (검증용)
done_m = set(read_csv_log(MART_LOG, ["key"])["key"].astype(str))
todo = [ym for ym in PERIOD_MONTHS if ym not in done_m
        or not (MART_B_DIR / f"mart_station_day_hour_{ym}.parquet").exists()
        or not (MART_A_DIR / f"mart_station_month_hour_{ym}.parquet").exists()]
print(f"대상 {N_MONTHS}개월 / 이번 처리 {len(todo)}개월")
for i, ym in enumerate(todo, 1):
    n_b, n_a = build_month(ym)
    print(f"[{i}/{len(todo)}] {ym}: 일별 {n_b:,}행 / 월별 {n_a:,}행")

## 4-5. 집계 검증 (통과해야 다음 단계로)
**왜:** 원본 = 대여 이벤트 + 예외, 대여 이벤트 = 정상 완료 + 반납 문제, 집계표 합계 = 로그 합계가 맞는지 확인해 잘못된 집계를 막습니다.

In [ ]:
slog = read_csv_log(SPLIT_LOG, ["key"])
ch = slog[slog["key"].astype(str).str.match(r"^c\d+$")]
raw_total, event_total, main_total = int(ch["raw"].sum()), int(ch["event"].sum()), int(ch["main"].sum())
return_issue_total, reject_total = int(ch["return_issue"].sum()), int(ch[REASONS].sum().sum())
assert raw_total == event_total + reject_total, "원본 = 대여 이벤트 + 예외 가 틀립니다."
assert event_total == main_total + return_issue_total, "대여 이벤트 = 정상 완료 + 반납 문제 가 틀립니다."

merge_log = LOG_DIR / "rental_merge_log.csv"
if merge_log.exists():
    assert raw_total == int(pd.read_csv(merge_log)["rows"].sum()), "병합 행 수와 처리 행 수가 다릅니다."

df_A = read_parquet_dir(MART_A_DIR)
agg_a = df_A[["rent_count", "completed_rent_count", "return_count"]].sum().astype("int64")
assert int(agg_a["rent_count"]) == event_total, "대여 이벤트 집계 불일치"
assert int(agg_a["completed_rent_count"]) == main_total, "완료 이용 집계 불일치"
assert int(agg_a["return_count"]) + late_return == main_total, "반납 집계 불일치"
sum_b = sum(pd.read_parquet(MART_B_DIR / f"mart_station_day_hour_{ym}.parquet",
                            columns=["rent_count", "completed_rent_count", "return_count"]).sum().astype("int64")
            for ym in PERIOD_MONTHS)
assert sum_b.equals(agg_a), "일별 표와 월별 표 합계가 다릅니다."
assert not df_A.duplicated(["station_id", "ym", "day_type", "hour"]).any()
assert df_A["ym"].nunique() == N_MONTHS, "집계가 빠진 달이 있습니다."

print(f"원본 {raw_total:,} = 대여 이벤트 {event_total:,} + 예외 {reject_total:,} ✔")
print(f"대여 이벤트 {event_total:,} = 정상 완료 {main_total:,} + 반납 문제 {return_issue_total:,} ✔")
print(f"기간 안 반납 {int(agg_a['return_count']):,} + 기간 뒤 반납 {late_return:,} = 정상 완료 ✔")
reason = ch[REASONS + RETURN_REASONS].sum().rename("rows").to_frame()
reason["pct_of_raw"] = (100 * reason["rows"] / raw_total).round(3)
display(reason)

# 5. 지하철 월별·시간대별 승하차 (`CardSubwayTime`)
## 5-1. API 수집 (월별)
**왜:** 역 × 월 × 시간대 승차·하차 인원으로 대여소 이용과의 관계를 봅니다. 한 달씩 받아 `api_raw/`에 저장합니다.

| 원본 컬럼 | 뜻 |
|---|---|
| `USE_MM` | 사용월 `YYYYMM` |
| `SBWY_ROUT_LN_NM`, `STTN` | 호선명, 역명 |
| `HR_{h}_GET_ON_NOPE`, `HR_{h}_GET_OFF_NOPE` | h시 승차·하차 인원 (월 합계) |

In [ ]:
card_frames, empty_months = [], []
for ym in PERIOD_MONTHS:
    df = api_cached("CardSubwayTime", f"subway_card_time_{ym}", ym)
    if df.empty:
        empty_months.append(ym)
    else:
        card_frames.append(df.assign(_query_period=ym))
raw_ct = concat_nonempty(card_frames, ignore_index=True)
assert len(raw_ct), "월별 승하차 API 응답이 비었습니다."
print(f"수집 {len(card_frames)}개월 / 자료 없음 {empty_months or '없음'}")

## 5-2. 중복 제거 · 세로(long) 변환
**왜:** 시간대가 컬럼으로 펼쳐진 표를 (월, 호선, 역, 시간)당 1행으로 바꿔 대여소 시간대 집계와 맞춥니다.

| 컬럼 | 뜻 |
|---|---|
| `ym`, `line`, `station`, `hour` | 연월, 호선, 역명, 시간 |
| `get_on`, `get_off` | 그 달 해당 시간 승차·하차 인원 합계 |

In [ ]:
df_ride_raw = raw_ct.rename(columns=str.lower).drop(columns=["_collected_at"], errors="ignore")
print("요청 월과 응답 월이 다른 행:", int((df_ride_raw["use_mm"] != df_ride_raw["_query_period"]).sum()), "(0이어야 정상)")
df_ride_raw = df_ride_raw[df_ride_raw["use_mm"].isin(PERIOD_MONTHS)]
hour_cols = [c for c in df_ride_raw.columns if re.fullmatch(r"hr_\d+_get_(on|off)_nope", c)]

key = ["use_mm", "sbwy_rout_ln_nm", "sttn"]
n_key, n_val = len(df_ride_raw.drop_duplicates(key)), len(df_ride_raw.drop_duplicates(key + hour_cols))
assert n_key == n_val, "값이 다른 중복 (월, 호선, 역)이 있습니다. 어느 행을 남길지 정해야 합니다."
df_ride = df_ride_raw.drop_duplicates(key).reset_index(drop=True)
print(f"시간대 컬럼 {len(hour_cols)}개 / 중복 제거 {len(df_ride_raw):,} → {len(df_ride):,}행 / {df_ride['use_mm'].nunique()}개월")

long = df_ride.melt(id_vars=key, value_vars=hour_cols, var_name="col", value_name="cnt")
hm = long["col"].str.extract(r"hr_(\d+)_get_(on|off)_nope")
long["hour"], long["kind"] = hm[0].astype("int64"), hm[1]
raw_cnt = long["cnt"].astype("string").str.strip()
cnt_num = pd.to_numeric(raw_cnt, errors="coerce")
bad = cnt_num.isna() & raw_cnt.notna() & (raw_cnt != "") & (raw_cnt != "\\N")
assert not bad.any(), "숫자로 읽을 수 없는 승하차 인원이 있습니다."
ALLOW_SUBWAY_MISSING_AS_ZERO = False     # 결측이 0명이라는 근거를 확인한 경우에만 True
if cnt_num.isna().any() and not ALLOW_SUBWAY_MISSING_AS_ZERO:
    raise ValueError(f"승하차 인원 결측 {int(cnt_num.isna().sum())}건 — 0명인지 확인 후 ALLOW_SUBWAY_MISSING_AS_ZERO 결정")
long["cnt"] = cnt_num.fillna(0)

df_ride_long = long.pivot_table(index=key + ["hour"], columns="kind", values="cnt", aggfunc="sum").reset_index()
df_ride_long.columns.name = None
df_ride_long = df_ride_long.rename(columns={"use_mm": "ym", "sbwy_rout_ln_nm": "line", "sttn": "station",
                                            "on": "get_on", "off": "get_off"})
del long
print(f"long 변환 {len(df_ride_long):,}행 (= {len(df_ride):,} × 24)")
display(df_ride_long.head())

## 5-3. 역 이름 → 물리역(`phys_key`) 매칭
**왜:** 승하차 자료에는 좌표가 없어 정규화한 역명(+호선)으로 3절의 물리역과 연결합니다. 이름이 바뀐 역은 수동으로 연결합니다.

| 컬럼 | 뜻 |
|---|---|
| `line`, `station` | 승하차 자료의 호선, 역명 |
| `phys_key` | 연결된 물리역 (실패면 결측) |
| `match_method` | `역명+호선` / `역명만` / `수동(개명)` |

In [ ]:
MANUAL_RENAME_FIX = {("7호선", "자양(뚝섬한강공원)"): "뚝섬유원지", ("4호선", "당고개"): "불암산"}
df_subway_pt = pd.read_parquet(OUT_DIR / "subway_station_point.parquet")


def match_station_names(sta):
    sta = sta.copy()
    sta["name_key"] = sta["station"].map(normalize_name)
    sta["line_key"] = sta["line"].map(normalize_line)
    point_keys = df_subway_pt[["name_key", "line_key", "phys_key"]].drop_duplicates(["name_key", "line_key"])
    out = sta.merge(point_keys, on=["name_key", "line_key"], how="left", validate="many_to_one")
    out["match_method"] = pd.Series(np.where(out["phys_key"].notna(), "역명+호선", None), dtype="string")
    # 2차: 역명만 같은 경우 (같은 이름의 물리역이 1곳일 때만)
    per_name = df_subway_pt.groupby("name_key")["phys_key"].nunique()
    name_to_phys = (df_subway_pt[df_subway_pt["name_key"].isin(per_name[per_name == 1].index)]
                    .drop_duplicates("name_key").set_index("name_key")["phys_key"])
    need = out["phys_key"].isna()
    out.loc[need, "phys_key"] = out.loc[need, "name_key"].map(name_to_phys)
    out.loc[need & out["phys_key"].notna(), "match_method"] = "역명만"
    valid_phys = set(df_subway_pt["phys_key"])
    for (ln, st), pk in MANUAL_RENAME_FIX.items():
        mm = (out["line"] == ln) & (out["station"] == st)
        if mm.any() and pk in valid_phys:
            out.loc[mm, ["phys_key", "match_method"]] = [pk, "수동(개명)"]
    out["phys_key"] = out["phys_key"].astype("string")
    return out


ride_match = match_station_names(df_ride_long[["line", "station"]].drop_duplicates().reset_index(drop=True))
save_parquet(ride_match, OUT_DIR / "subway_name_map_month.parquet")
display(ride_match["match_method"].fillna("실패").value_counts().to_frame("역 수"))
print("매칭 실패 (서울 밖 역이면 무시, 서울 안 역이면 MANUAL_RENAME_FIX에 추가):")
display(ride_match[ride_match["phys_key"].isna()])

## 5-4. 물리역 단위 합산 (`subway_ride_month_hour`)
**왜:** 환승역의 호선별 인원을 합쳐 물리역 × 연월 × 시간 1행으로 만들어 대여소와 연결합니다.

| 컬럼 | 뜻 |
|---|---|
| `phys_key`, `ym`, `hour` | 물리역, 연월, 시간 (키) |
| `get_on`, `get_off` | 그 달 해당 시간 승차·하차 인원 합계 |

In [ ]:
df_ride_phys = df_ride_long.merge(ride_match[["line", "station", "phys_key"]], on=["line", "station"],
                                  how="left", validate="many_to_one")
print("매칭 실패로 제외:", f"{int(df_ride_phys['phys_key'].isna().sum()):,}행")
df_subway_ride = (df_ride_phys.dropna(subset=["phys_key"])
                  .groupby(["phys_key", "ym", "hour"], as_index=False)[["get_on", "get_off"]].sum())
df_subway_ride = df_subway_ride.astype({"phys_key": "string", "ym": "string", "hour": "int64"})
df_subway_ride["get_on"]  = df_subway_ride["get_on"].round().astype("int64")
df_subway_ride["get_off"] = df_subway_ride["get_off"].round().astype("int64")
assert not df_subway_ride.duplicated(["phys_key", "ym", "hour"]).any()
save_parquet(df_subway_ride, OUT_DIR / "subway_ride_month_hour.parquet")
print(f"물리역 {df_subway_ride['phys_key'].nunique()}곳 / {len(df_subway_ride):,}행")

# 6. 서울교통공사 일별·시간대별 승하차 (`subwSttn`)
**왜:** 월별 자료는 평일·주말을 나눌 수 없어, 서울교통공사 일별 실측으로 요일별 모양(프로필)을 만들고 월 합계를 날짜별로 나눕니다(추정은 `is_estimated=True`).

- 이 API는 날짜 조건이 무시되므로 **전체를 1회** 받고(5만 행마다 저장, 끊겨도 이어서) 날짜로 기간을 거릅니다.
- 1~8호선만, 제공 기간과 분석 기간이 겹치는 날만 실측입니다. 실측이 없으면 6절은 건너뛰고 7·8절은 월별 자료만 씁니다.

## 6-1. API 수집

| 원본 컬럼 | 뜻 |
|---|---|
| `MVMN_YMD` | 수송일자 |
| `LINE`, `STTN_NO`, `STTN` | 호선, 역번호, 역명 |
| `GTNF_SE` | `승차` / `하차` |
| `HR06_BFR`, `HR06`~`HR23`, `HR24` | 06시 이전 합계, 각 시간, 24시 이후 합계 |

In [ ]:
METRO_DIR        = API_RAW_DIR / "subway_metro_daily_hour"
METRO_LOG        = LOG_DIR / "metro_fetch_log.csv"
METRO_BATCH_ROWS = 50_000


def fetch_metro():
    log = read_csv_log(METRO_LOG, ["key", "rows", "next"])
    if "ALL" in set(log["key"]):
        print("서울교통공사 일별: 이미 수집 → 건너뜀")
        return
    state = {"buf": [], "start": int(log["next"].max()) if len(log) else 1}

    def flush(next_start):
        df = pd.concat(state["buf"], ignore_index=True)
        save_parquet(to_text_frame(df).assign(_collected_at=now_str()), METRO_DIR / f"part_{state['start']:07d}.parquet")
        append_csv_log(METRO_LOG, {"key": f"rows_{state['start']:07d}", "rows": len(df), "next": next_start})
        state["buf"], state["start"] = [], next_start

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            for _, p_end, page, total in iter_pages("subwSttn", "", start=state["start"]):
                state["buf"].append(page)
                if sum(map(len, state["buf"])) >= METRO_BATCH_ROWS:
                    flush(p_end + 1)
                    print(f"  {p_end:,} / {total}행 수신")
            break
        except ApiRequestError:
            state["buf"] = []                         # 마지막 저장 지점부터 다시
            print(f"[요청 실패] {attempt}/{MAX_RETRIES}회")
            if attempt == MAX_RETRIES:
                raise
            time.sleep(attempt * 3)
    if state["buf"]:
        flush(state["start"] + sum(map(len, state["buf"])))
    append_csv_log(METRO_LOG, {"key": "ALL", "rows": int(read_csv_log(METRO_LOG, ["rows"])["rows"].sum()), "next": 0})


fetch_metro()
raw_metro = read_parquet_dir(METRO_DIR)
if len(raw_metro):
    _d = pd.to_datetime(raw_metro["MVMN_YMD"].astype("string").str.slice(0, 10), errors="coerce")
    print(f"전체 {len(raw_metro):,}행 ({_d.min():%Y-%m-%d} ~ {_d.max():%Y-%m-%d}) → 분석 기간 안 {int(_d.between(START_TS, END_TS).sum()):,}행")
    raw_metro = raw_metro[_d.between(START_TS, END_TS)].reset_index(drop=True)
METRO_OK = len(raw_metro) > 0
if not METRO_OK:
    print("⚠ 분석 기간에 서울교통공사 일별 실측이 없습니다 → 6-2~6-8은 건너뜁니다.")

## 6-2. 중복 제거 · 역 이름 매칭
**왜:** 같은 (날짜, 호선, 역번호, 승하차)는 1행만 남기고(값이 다른 중복이면 중단), 5-3과 같은 규칙으로 물리역에 연결합니다.

In [ ]:
HR_COLS = ["hr06_bfr"] + [f"hr{h:02d}" for h in range(6, 25)]
KEY_M   = ["mvmn_ymd", "line", "sttn_no", "gtnf_se"]
if METRO_OK:
    df_metro_raw = raw_metro.rename(columns=str.lower).drop(columns=["_collected_at"], errors="ignore")
    for c in ["mvmn_ymd", "line", "sttn_no", "sttn", "gtnf_se"]:
        df_metro_raw[c] = df_metro_raw[c].astype("string").str.strip()
    n_key, n_val = len(df_metro_raw.drop_duplicates(KEY_M)), len(df_metro_raw.drop_duplicates(KEY_M + HR_COLS))
    if n_key != n_val:
        raise ValueError(f"값이 다른 중복 키 {n_val - n_key:,}건 — 출처·버전 확인 필요")
    df_metro_clean = df_metro_raw.sort_values(KEY_M).drop_duplicates(KEY_M).reset_index(drop=True)
    df_metro_clean["mvmn_ymd"] = as_date(pd.to_datetime(df_metro_clean["mvmn_ymd"], errors="coerce"))
    for c in HR_COLS:
        df_metro_clean[c] = to_float(df_metro_clean[c])
    name_map = match_station_names(df_metro_clean[["line", "sttn"]].drop_duplicates()
                                   .rename(columns={"sttn": "station"}).reset_index(drop=True))
    save_parquet(name_map[["line", "station", "phys_key", "match_method"]], OUT_DIR / "subway_name_map_daily_metro.parquet")
    print(f"원본 {len(df_metro_raw):,} → 정리 {len(df_metro_clean):,}행")
    display(name_map["match_method"].fillna("실패").value_counts().to_frame("역 수"))

## 6-3. 물리역 × 날짜 × 시간 (`subway_ride_day_hour`)
**왜:** 시간대 컬럼을 행으로 풀고 물리역 단위로 합쳐 실측 일별 표를 만듭니다. `hour=-1`(06시 이전), `24`(24시 이후)는 쪼갤 수 없는 덩어리 시간입니다.

| 컬럼 | 뜻 |
|---|---|
| `phys_key`, `ride_date`, `hour` | 물리역, 날짜, 시간(-1, 6~24) |
| `get_on`, `get_off` | 승차·하차 인원 |
| `is_estimated` | 추정 여부 (실측이라 False) |

In [ ]:
if METRO_OK:
    HOUR_OF = {"hr06_bfr": -1, **{f"hr{h:02d}": h for h in range(6, 25)}}
    ml = df_metro_clean.melt(id_vars=["mvmn_ymd", "line", "sttn", "gtnf_se"], value_vars=HR_COLS, var_name="col", value_name="val")
    ml["hour"] = ml["col"].map(HOUR_OF).astype("int64")
    ml = ml.merge(name_map[["line", "station", "phys_key"]].rename(columns={"station": "sttn"}),
                  on=["line", "sttn"], how="left", validate="many_to_one").dropna(subset=["phys_key"])
    pv = ml.groupby(["phys_key", "mvmn_ymd", "hour", "gtnf_se"])["val"].sum(min_count=1).unstack("gtnf_se").reset_index()
    pv.columns.name = None
    df_day_hour = pv.rename(columns={"mvmn_ymd": "ride_date", "승차": "get_on", "하차": "get_off"})
    for c in ["get_on", "get_off"]:
        df_day_hour[c] = df_day_hour[c].round().astype("Int64") if c in df_day_hour else pd.array([pd.NA] * len(df_day_hour), dtype="Int64")
    df_day_hour["phys_key"]     = df_day_hour["phys_key"].astype("string")
    df_day_hour["ride_date"]    = as_date(df_day_hour["ride_date"])
    df_day_hour["is_estimated"] = False
    df_day_hour = df_day_hour[["phys_key", "ride_date", "hour", "get_on", "get_off", "is_estimated"]]
    assert not df_day_hour.duplicated(["phys_key", "ride_date", "hour"]).any()
    save_parquet(df_day_hour, OUT_DIR / "subway_ride_day_hour.parquet")
    del ml, pv
    print(f"{len(df_day_hour):,}행 / 물리역 {df_day_hour['phys_key'].nunique()}곳 / "
          f"{df_day_hour['ride_date'].min():%Y-%m-%d} ~ {df_day_hour['ride_date'].max():%Y-%m-%d}")

## 6-4. 검증 — 시간대 정렬 · 종합 점검
**왜:** 일별 표와 월별 표의 같은 시간 비율이 고르게 나오는지(1시간 밀림 방지)와 날짜·역 누락을 확인합니다.

In [ ]:
if METRO_OK:
    df_day_hour["ym"] = df_day_hour["ride_date"].dt.strftime("%Y%m")
    measured_months = sorted(df_day_hour["ym"].unique())
    CHK_YM = measured_months[0]
    keys_chk = df_day_hour.loc[df_day_hour["ym"] == CHK_YM, "phys_key"].unique()
    d_on = df_day_hour[(df_day_hour["ym"] == CHK_YM) & df_day_hour["hour"].between(6, 23)].groupby("hour")["get_on"].sum()
    m_on = df_subway_ride[(df_subway_ride["ym"] == CHK_YM) & df_subway_ride["phys_key"].isin(keys_chk)].groupby("hour")["get_on"].sum()
    df_chk = pd.DataFrame({"hour": d_on.index, "on_daily": d_on.values.astype(float)})
    for name, sh in [("shift0", 0), ("shift_m1", -1), ("shift_p1", 1)]:
        df_chk[name] = (df_chk["on_daily"] / df_chk["hour"].add(sh).map(m_on).astype(float)).round(3)
    cv = (df_chk[["shift0", "shift_m1", "shift_p1"]].std() / df_chk[["shift0", "shift_m1", "shift_p1"]].mean()).round(3)
    print(f"비교 연월 {CHK_YM} / 변동계수(작을수록 일정):", cv.to_dict())
    print("판정:", "시간대 정의 일치 ✔" if cv.idxmin() == "shift0" else f"⚠ {cv.idxmin()}이 더 일정 → 시간 정의 확인")

    results = []
    def judge(name, ok, detail=""):
        results.append((name, "PASS" if ok else "WARN", detail))
    n_days_meas = df_day_hour["ride_date"].nunique()
    exp_days = (df_day_hour["ride_date"].max() - df_day_hour["ride_date"].min()).days + 1
    judge("일별 매핑 실패 0곳", name_map["phys_key"].notna().all(), f"{int(name_map['phys_key'].isna().sum())}곳")
    judge(f"날짜 연속 ({exp_days}일)", n_days_meas == exp_days, f"{n_days_meas}일")
    hcount = df_day_hour.groupby(["phys_key", "ride_date"]).size().value_counts()
    judge("(역·날짜)당 시간 행 20개", list(hcount.index) == [20], str(hcount.to_dict()))
    judge("음수 0건", int(((df_day_hour["get_on"] < 0) | (df_day_hour["get_off"] < 0)).sum()) == 0)
    dm = df_day_hour[df_day_hour["hour"].between(6, 23)].groupby(["phys_key", "ym"])["get_on"].sum().rename("on_d")
    mm_ = df_subway_ride[df_subway_ride["hour"].between(6, 23)].groupby(["phys_key", "ym"])["get_on"].sum().rename("on_m")
    df_ratio = pd.concat([dm, mm_], axis=1, join="inner").groupby(level="ym").sum()
    df_ratio["ratio"] = (df_ratio["on_d"].astype(float) / df_ratio["on_m"]).round(3)
    judge("월별 일별÷월별 비율 0.90~0.97", df_ratio["ratio"].between(0.90, 0.97).all(),
          f"{df_ratio['ratio'].min():.3f} ~ {df_ratio['ratio'].max():.3f}")
    display(df_ratio)
    display(pd.DataFrame(results, columns=["항목", "판정", "상세"]))

## 6-5. 커버리지 (`subway_daily_coverage`)
**왜:** 대여소와 연결된 역마다 일별 실측이 월별 인원의 몇 %를 담는지 보고 실측을 그대로 쓸지, 모양만 쓸지 정합니다.

| 컬럼 | 뜻 |
|---|---|
| `phys_key` | 물리역 |
| `n_nearest_bike` | 이 역이 '가장 가까운 역'인 대여소 수 |
| `months_overlap` | 일별·월별이 함께 있는 달 수 |
| `monthly_total`, `ratio` | 월별 인원 합(06~23시), 일별 ÷ 월별 |
| `coverage_class` | `full`(88% 이상) / `partial`(환승역 일부 노선만) / `no_daily`(실측 없음, 프로필로 추정) |

In [ ]:
df_link = pd.read_parquet(OUT_DIR / "station_subway_link.parquet")
if METRO_OK:
    vd = (df_day_hour[df_day_hour["hour"].between(6, 23)].assign(v=lambda x: x["get_on"].fillna(0) + x["get_off"].fillna(0))
          .groupby(["phys_key", "ym"])["v"].sum().rename("v_d"))
    vm = (df_subway_ride[df_subway_ride["hour"].between(6, 23)].assign(v=lambda x: x["get_on"] + x["get_off"])
          .groupby(["phys_key", "ym"])["v"].sum().rename("v_m"))
    agg = (pd.concat([vd, vm], axis=1, join="inner").reset_index()
           .groupby("phys_key").agg(months_overlap=("ym", "size"), monthly_total=("v_m", "sum"), daily_total=("v_d", "sum")))
    agg["ratio"] = (agg["daily_total"].astype(float) / agg["monthly_total"].replace(0, np.nan)).round(3)
    cov = (df_link.groupby("phys_key").agg(n_nearest_bike=("is_nearest", "sum")).reset_index()
           .merge(agg.reset_index(), on="phys_key", how="left", validate="one_to_one"))
    cov["months_overlap"] = cov["months_overlap"].fillna(0).astype("int64")
    cov["coverage_class"] = pd.Series(np.where(cov["months_overlap"] == 0, "no_daily",
                                               np.where(cov["ratio"] >= 0.88, "full", "partial")), dtype="string")
    cov["n_nearest_bike"] = cov["n_nearest_bike"].astype("int64")
    df_cov = cov[["phys_key", "n_nearest_bike", "months_overlap", "monthly_total", "ratio", "coverage_class"]]
    save_parquet(df_cov, OUT_DIR / "subway_daily_coverage.parquet")
    display(df_cov.groupby("coverage_class").agg(stations=("phys_key", "size"), bike_stations=("n_nearest_bike", "sum")))

## 6-6. 요일유형별 모양 (`subway_day_profile_station`, `subway_day_profile_global`)
**왜:** 실측 기간의 역 × 시간 × 요일구분 하루 평균으로, 실측이 없는 날짜·역의 월 합계를 나눌 비중을 만듭니다.

| 컬럼 | 뜻 |
|---|---|
| `phys_key` | 물리역 (global 표에는 없음 = `full` 역 전체 합) |
| `hour`, `day_type` | 시간(0~23), 요일구분 |
| `f_on`, `f_off` | 하루 평균 승차·하차 (비중 계산용) |

- 덩어리 시간 대응: 06시 이전(-1) → 4·5시 / 24시 이후(24) → 0~3시

In [ ]:
if METRO_OK:
    HM = pd.DataFrame([(-1, 4), (-1, 5)] + [(24, h) for h in range(0, 4)] + [(h, h) for h in range(6, 24)], columns=["src", "dst"])
    MIN_OVERLAP_MONTHS = math.ceil(len(measured_months) * 20 / 24)
    meas_cal = cal[cal["cal_date"].between(df_day_hour["ride_date"].min(), df_day_hour["ride_date"].max())][["cal_date", "day_type"]]
    nd = meas_cal.groupby("day_type").size()
    good = df_cov.loc[df_cov["coverage_class"].isin(["full", "partial"]) & (df_cov["months_overlap"] >= MIN_OVERLAP_MONTHS), "phys_key"]
    meas = (df_day_hour[df_day_hour["phys_key"].isin(good)]
            .merge(HM, left_on="hour", right_on="src", how="inner").drop(columns=["hour", "src"]).rename(columns={"dst": "hour"})
            .merge(meas_cal, left_on="ride_date", right_on="cal_date", how="inner", validate="many_to_one"))
    prof_st = meas.groupby(["phys_key", "hour", "day_type"], as_index=False).agg(s_on=("get_on", "sum"), s_off=("get_off", "sum"))
    prof_st["f_on"]  = prof_st["s_on"].astype(float) / prof_st["day_type"].map(nd)
    prof_st["f_off"] = prof_st["s_off"].astype(float) / prof_st["day_type"].map(nd)
    prof_st = prof_st[["phys_key", "hour", "day_type", "f_on", "f_off"]]
    full_keys = df_cov.loc[df_cov["coverage_class"] == "full", "phys_key"]
    prof_gl = prof_st[prof_st["phys_key"].isin(full_keys)].groupby(["hour", "day_type"], as_index=False)[["f_on", "f_off"]].sum()
    save_parquet(prof_st, OUT_DIR / "subway_day_profile_station.parquet")
    save_parquet(prof_gl, OUT_DIR / "subway_day_profile_global.parquet")
    print(f"MIN_OVERLAP_MONTHS = {MIN_OVERLAP_MONTHS} / 역별 프로필 {prof_st['phys_key'].nunique()}곳")
    display(prof_gl[prof_gl["hour"].isin([8, 18])].round(0).sort_values(["hour", "day_type"]))

## 6-7. 일별 최종 표 (`subway_ride_day_hour_full`)
**왜:** 월 합계 × 그 날의 비중으로 연결된 역의 모든 날짜·시간 값을 만듭니다 (월 합계는 그대로 유지).

| 컬럼 | 뜻 |
|---|---|
| `phys_key`, `ride_date`, `hour` | 물리역, 날짜, 시간(0~23) |
| `get_on`, `get_off` | 승차·하차 (월 합계 × 비중) |
| `shape_source` | `measured`(실측 비중) / `allocated_range`(덩어리 시간 배분) / `profile`(요일 프로필) |
| `is_estimated` | 날짜·시간 분포가 추정이면 True |

In [ ]:
def build_full_month(ym, use_meas=True, profile_station=True):
    d0 = pd.Timestamp(f"{ym[:4]}-{ym[4:]}-01")
    d1 = d0 + pd.offsets.MonthBegin(1)
    days = cal[(cal["cal_date"] >= d0) & (cal["cal_date"] < d1)][["cal_date", "day_type"]]
    mm = df_subway_ride[(df_subway_ride["ym"] == ym) & df_subway_ride["phys_key"].isin(df_cov["phys_key"])]
    mm = mm.rename(columns={"get_on": "m_on", "get_off": "m_off"})[["phys_key", "hour", "m_on", "m_off"]]
    if mm.empty:
        return pd.DataFrame()
    g = mm.merge(days, how="cross")
    me = (df_day_hour[(df_day_hour["ride_date"] >= d0) & (df_day_hour["ride_date"] < d1)]
          .merge(HM, left_on="hour", right_on="src").drop(columns=["hour", "src"]).rename(columns={"dst": "hour"}))
    me = me[me["phys_key"].isin(df_cov["phys_key"])][["phys_key", "ride_date", "hour", "get_on", "get_off"]]
    mt = me.groupby(["phys_key", "hour"]).agg(t_on=("get_on", "sum"), t_off=("get_off", "sum"), n=("ride_date", "size")).reset_index()
    mt = mt[mt["n"] == len(days)].drop(columns="n")                     # 그 달 모든 날짜에 실측이 있을 때만
    g = (g.merge(me.rename(columns={"ride_date": "cal_date", "get_on": "v_on", "get_off": "v_off"}),
                 on=["phys_key", "hour", "cal_date"], how="left", validate="one_to_one")
          .merge(mt, on=["phys_key", "hour"], how="left", validate="many_to_one"))
    if profile_station:
        g = g.merge(prof_st.rename(columns={"f_on": "ps_on", "f_off": "ps_off"}), on=["phys_key", "hour", "day_type"],
                    how="left", validate="many_to_one")
    else:
        g["ps_on"] = g["ps_off"] = np.nan
    g = g.merge(prof_gl.rename(columns={"f_on": "pg_on", "f_off": "pg_off"}), on=["hour", "day_type"], how="left", validate="many_to_one")
    g["f_on"]  = g["ps_on"].astype(float).fillna(g["pg_on"].astype(float))
    g["f_off"] = g["ps_off"].astype(float).fillna(g["pg_off"].astype(float))
    use_m = use_meas & (g["t_on"].astype(float) > 0) & (g["t_off"].astype(float) > 0)
    grp = g.groupby(["phys_key", "hour"])
    s_on, s_off, n_d = grp["f_on"].transform("sum"), grp["f_off"].transform("sum"), grp["f_on"].transform("size")
    w_on  = np.where(use_m, g["v_on"].astype(float) / g["t_on"].astype(float),
                     np.where(s_on > 0, g["f_on"] / s_on.replace(0, np.nan), 1.0 / n_d))
    w_off = np.where(use_m, g["v_off"].astype(float) / g["t_off"].astype(float),
                     np.where(s_off > 0, g["f_off"] / s_off.replace(0, np.nan), 1.0 / n_d))
    early = g["hour"].between(0, 5)                                      # 덩어리 시간에서 나눈 값
    return pd.DataFrame({
        "phys_key": g["phys_key"].astype("string"), "ride_date": g["cal_date"], "hour": g["hour"].astype("int64"),
        "get_on": (g["m_on"] * w_on).round(2), "get_off": (g["m_off"] * w_off).round(2),
        "shape_source": pd.Series(np.where(use_m & ~early, "measured", np.where(use_m & early, "allocated_range", "profile")),
                                  index=g.index, dtype="string"),
        "is_estimated": (~use_m | early).to_numpy()})


if METRO_OK:
    df_full = concat_nonempty([build_full_month(ym) for ym in sorted(df_subway_ride["ym"].unique())], ignore_index=True)
    assert not df_full.duplicated(["phys_key", "ride_date", "hour"]).any()
    save_parquet(df_full, OUT_DIR / "subway_ride_day_hour_full.parquet")
    print(f"{len(df_full):,}행 / 물리역 {df_full['phys_key'].nunique()}곳")
    display(df_full["shape_source"].value_counts().to_frame("행 수"))

## 6-8. 검증 — 월 합계 일치 · 백테스트
**왜:** 일별 표의 월 합계가 월별 표와 같은지(비율 1.000), 실측이 있는 달을 일부러 프로필로만 만들어 추정 오차를 확인합니다. 오차가 10%를 넘으면 추정 일별 값 대신 월 평균 변수만 쓰세요.

In [ ]:
if METRO_OK:
    a_ = df_full.assign(ym=df_full["ride_date"].dt.strftime("%Y%m")).groupby("ym")[["get_on", "get_off"]].sum()
    b_ = df_subway_ride[df_subway_ride["phys_key"].isin(df_cov["phys_key"])].groupby("ym")[["get_on", "get_off"]].sum()
    ratio = (a_ / b_).round(4)
    print("월 합계 비율 (승차):", ratio["get_on"].min(), "~", ratio["get_on"].max(), "/ (하차):", ratio["get_off"].min(), "~", ratio["get_off"].max())

    BACKTEST_YM = "202406" if "202406" in measured_months else measured_months[len(measured_months) // 2]
    base_bt = (df_full[(df_full["ride_date"].dt.strftime("%Y%m") == BACKTEST_YM) & df_full["hour"].between(7, 9)
                       & (df_full["shape_source"] == "measured")].groupby(["phys_key", "ride_date"])["get_on"].sum())
    rows = []
    for label, ps in [("역별 프로필", True), ("전체 프로필만", False)]:
        bt = build_full_month(BACKTEST_YM, use_meas=False, profile_station=ps)
        bt = bt[bt["hour"].between(7, 9)].groupby(["phys_key", "ride_date"])["get_on"].sum()
        jj = pd.concat([base_bt.rename("a"), bt.rename("b")], axis=1, join="inner")
        err = (jj["b"] - jj["a"]).abs() / jj["a"].replace(0, np.nan)
        rows.append({"방식": label, "station_days": len(jj), "mean_abs_pct_error": round(err.mean(), 3),
                     "median_abs_pct_error": round(err.median(), 3)})
    print(f"백테스트 연월 {BACKTEST_YM} (출근 7~9시 승차)")
    display(pd.DataFrame(rows))

# 7. 통합 변수 — 지하철 × 따릉이
## 7-1. 대여소별 지하철 고정 변수 (`feature_station_subway`)
**왜:** 대여소 1곳 = 1행으로 가장 가까운 역·거리·역세권 여부를 정리해 EDA·모델에서 바로 붙입니다.

| 컬럼 | 뜻 |
|---|---|
| `station_id`, `station_name`, `gu` | 대여소 ID, 이름, 자치구 |
| `hold_num`, `sta_lat`, `sta_long` | 거치대 수, 위도, 경도 (위경도 뒤바뀜 교정 후) |
| `has_coord` | 좌표 정상 여부 (False면 거리 변수는 '알 수 없음' 결측) |
| `nearest_phys_key`, `nearest_subway_name`, `nearest_lines` | 500m 안 가장 가까운 역 |
| `nearest_distance_m` | 그 역까지 거리(m, 500m 안에 역이 없으면 결측) |
| `is_subway_near_300` | 역세권(300m 안 역 있음) |
| `n_subway_300`, `n_subway_500` | 300m / 500m 안 역 수 |

In [ ]:
df_both = pd.read_parquet(OUT_DIR / "bike_station_both.parquet")
df_diff = pd.read_parquet(OUT_DIR / "bike_station_diff.parquet")
df_subway_ride = pd.read_parquet(OUT_DIR / "subway_ride_month_hour.parquet")

# 전체 대여소 = both + diff (같은 ID가 양쪽에 있으면 both 우선)
base = (pd.concat([df_both.assign(pri=1), df_diff.drop(columns="data_source").assign(pri=2)], ignore_index=True)
          .sort_values(["rent_id", "pri"]).drop_duplicates("rent_id").reset_index(drop=True))
swap_fix(base, "sta_lat", "sta_long")                                   # 3-2와 같은 위경도 뒤바뀜 교정
base["has_coord"] = valid_coord(base["sta_lat"], base["sta_long"]).fillna(False).astype(bool)
nr  = df_link[df_link["is_nearest"]][["rent_id", "phys_key", "subway_name", "lines", "distance_m"]]
cnt = (df_link.assign(w300=df_link["distance_m"] <= NEAR_RADIUS_M)
       .groupby("rent_id").agg(n_subway_300=("w300", "sum"), n_subway_500=("phys_key", "size")).reset_index())
fs = base.merge(nr, on="rent_id", how="left", validate="one_to_one").merge(cnt, on="rent_id", how="left", validate="one_to_one")
feature_station_subway = pd.DataFrame({
    "station_id": fs["rent_id"].astype("string"), "station_name": fs["rent_nm"].astype("string"), "gu": fs["sta_loc"].astype("string"),
    "hold_num": fs["hold_num"].astype("Int64"), "sta_lat": fs["sta_lat"], "sta_long": fs["sta_long"], "has_coord": fs["has_coord"],
    "nearest_phys_key": fs["phys_key"].astype("string"), "nearest_subway_name": fs["subway_name"].astype("string"),
    "nearest_lines": fs["lines"].astype("string"), "nearest_distance_m": fs["distance_m"],
    "is_subway_near_300": pd.array(np.where(fs["has_coord"], (fs["distance_m"] <= NEAR_RADIUS_M).fillna(False), pd.NA), dtype="boolean"),
    "n_subway_300": pd.array(np.where(fs["has_coord"], fs["n_subway_300"].fillna(0), pd.NA), dtype="Int64"),
    "n_subway_500": pd.array(np.where(fs["has_coord"], fs["n_subway_500"].fillna(0), pd.NA), dtype="Int64"),
})
assert feature_station_subway["station_id"].is_unique
save_parquet(feature_station_subway, OUT_DIR / "feature_station_subway.parquet")
display(pd.DataFrame({"대여소": [len(feature_station_subway)], "좌표 있음": [int(feature_station_subway["has_coord"].sum())],
                      "역세권(300m)": [int(feature_station_subway["is_subway_near_300"].sum())],
                      "500m 안 역 있음": [int(feature_station_subway["nearest_phys_key"].notna().sum())]}))

## 7-2. 대여소 × 연월 × 시간 지하철 변수 (`feature_station_subway_month_hour`)
**왜:** 대여소마다 인근 역의 시간대별 하루 평균 승하차를 붙입니다. 같은 달 값은 그 달이 끝나야 알 수 있으므로 모델에서는 **전월 값**만 씁니다(8절).

| 컬럼 | 뜻 |
|---|---|
| `station_id`, `ym`, `hour` | 대여소, 연월, 시간 (키) |
| `nearest_on_pd`, `nearest_off_pd` | 가장 가까운 역 하루 평균 승차·하차 |
| `w300_on_pd`, `w300_off_pd` | 300m 안 모든 역 합계 하루 평균 (300m 안 역 없으면 결측) |
| `days_in_month` | 그 달 일수 |

In [ ]:
jx = df_link[["rent_id", "phys_key", "is_nearest", "distance_m"]].merge(df_subway_ride, on="phys_key", how="inner")   # 의도된 다대다
days_ym = cal.groupby("ym").size().rename("days_in_month")
kk = ["rent_id", "ym", "hour"]
near = jx[jx["is_nearest"]].groupby(kk)[["get_on", "get_off"]].max().rename(columns={"get_on": "nearest_on", "get_off": "nearest_off"})
w300 = jx[jx["distance_m"] <= NEAR_RADIUS_M].groupby(kk)[["get_on", "get_off"]].sum().rename(columns={"get_on": "w300_on", "get_off": "w300_off"})
fmh = jx[kk].drop_duplicates().set_index(kk).join(near).join(w300).reset_index()
fmh = fmh.merge(days_ym, left_on="ym", right_index=True, how="left", validate="many_to_one")
for src, dst in [("nearest_on", "nearest_on_pd"), ("nearest_off", "nearest_off_pd"), ("w300_on", "w300_on_pd"), ("w300_off", "w300_off_pd")]:
    fmh[dst] = (fmh[src] / fmh["days_in_month"]).round(2)
feature_station_subway_month_hour = fmh.rename(columns={"rent_id": "station_id"})[
    ["station_id", "ym", "hour", "nearest_on_pd", "nearest_off_pd", "w300_on_pd", "w300_off_pd", "days_in_month"]]
feature_station_subway_month_hour["days_in_month"] = feature_station_subway_month_hour["days_in_month"].astype("int64")
assert not feature_station_subway_month_hour.duplicated(["station_id", "ym", "hour"]).any()
save_parquet(feature_station_subway_month_hour, OUT_DIR / "feature_station_subway_month_hour.parquet")
del jx
print(f"{len(feature_station_subway_month_hour):,}행 / 대여소 {feature_station_subway_month_hour['station_id'].nunique():,}곳 / "
      f"{feature_station_subway_month_hour['ym'].nunique()}개월")

## 7-3. 검증 — 역세권 vs 비역세권 대여량
**왜:** 새 변수표로 계산해도 역세권 대여소의 하루 평균 대여가 더 많은지(시각화 노트북 2-3과 같은 방향) 빠르게 확인합니다.

In [ ]:
st_chk = (df_A[smatch(df_A["station_id"], r"^ST-")].groupby("station_id")["rent_count"].sum()
          .div(DAYS_IN_PERIOD).rename("rent_all").reset_index())
display(st_chk.merge(feature_station_subway, on="station_id", how="inner", validate="one_to_one")
        .groupby("is_subway_near_300", dropna=False).agg(n_stations=("station_id", "size"), avg_rent_all=("rent_all", "mean")).round(1))

# 8. 모델링 데이터 (`model_station_day_hour/`)
**왜:** 수요예측 기본 단위(대여소 × 날짜 × 시간)에 달력·시간대·대여소·**전월** 지하철 변수를 붙입니다. 이용이 없던 시간은 대여소 활동 기간(첫~마지막 이용일) 안에서만 0으로 채웁니다.

| 변수 | 모델 입력 | 이유 |
|---|---|---|
| 달력·공휴일·시간대 | ✅ | 미리 알 수 있음 |
| 대여소 고정 속성 (구·좌표·거치대·역 거리) | ✅ (주의) | 수집 시점 스냅샷 |
| 전월 지하철 (`prev_month_*`) | ✅ | 같은 달 값은 누수 → 한 달 뒤로 이동 |
| `return_count`, `net_flow`, `completed_rent_count` | ❌ | 타깃과 같은 시점의 결과 |

> 학습/검증은 **시간 순서로** 나누세요 (예: 2024-01~2025-12 학습, 2026-01~06 검증).

In [ ]:
MODEL_DIR = OUT_DIR / "model_station_day_hour"
B_FILES = {f.stem.rsplit("_", 1)[-1]: f for f in sorted(MART_B_DIR.glob("*.parquet"))}

# 1) 대여소 활동 기간 (첫·마지막 이용일)
span = (pd.concat([pd.read_parquet(f, columns=["station_id", "d"]).groupby("station_id")["d"].agg(["min", "max"]) for f in B_FILES.values()])
        .groupby(level=0).agg(first_d=("min", "min"), last_d=("max", "max")).reset_index())
span = span[smatch(span["station_id"], r"^ST-")]
span["first_d"], span["last_d"] = as_date(span["first_d"]), as_date(span["last_d"])

# 2) 고정 변수 · 달력 · 전월 지하철 변수 (ym을 한 달 뒤로 밀어 다음 달 행에 붙임)
static = feature_station_subway[["station_id", "gu", "hold_num", "sta_lat", "sta_long", "has_coord",
                                 "nearest_distance_m", "is_subway_near_300", "n_subway_300", "n_subway_500"]]
cal_feat = cal[["cal_date", "ym", "year", "month", "dow", "is_weekend", "is_holiday", "day_type"]].rename(columns={"cal_date": "date"})
prev = feature_station_subway_month_hour.copy()
prev["ym"] = (pd.to_datetime(prev["ym"], format="%Y%m") + pd.offsets.MonthBegin(1)).dt.strftime("%Y%m").astype("string")
prev = prev.rename(columns={c: f"prev_month_{c}" for c in ["nearest_on_pd", "nearest_off_pd", "w300_on_pd", "w300_off_pd"]}).drop(columns="days_in_month")
print(f"대여소 {len(span):,}곳 / 대여소 표에 없는 대여소 {len(set(span['station_id']) - set(static['station_id']))}곳")


def build_model_month(ym):
    d0 = pd.Timestamp(f"{ym[:4]}-{ym[4:]}-01")
    d1 = d0 + pd.offsets.MonthBegin(1) - pd.Timedelta(days=1)
    act = span[(span["first_d"] <= d1) & (span["last_d"] >= d0)]
    dates = pd.DataFrame({"date": as_date(pd.Series(pd.date_range(max(d0, START_TS), min(d1, END_TS), freq="D")))})
    grid = (act[["station_id", "first_d", "last_d"]].merge(dates, how="cross")
            .merge(pd.DataFrame({"hour": np.arange(24, dtype="int64")}), how="cross"))
    grid = grid[(grid["date"] >= grid["first_d"]) & (grid["date"] <= grid["last_d"])].drop(columns=["first_d", "last_d"])
    b = pd.read_parquet(B_FILES[ym], columns=["station_id", "d", "hour", "rent_count", "completed_rent_count", "return_count"]).rename(columns={"d": "date"})
    b = b[smatch(b["station_id"], r"^ST-")]
    b["date"] = as_date(b["date"])
    n0 = len(grid)
    vals = ["rent_count", "completed_rent_count", "return_count"]
    mdl = grid.merge(b, on=["station_id", "date", "hour"], how="left", validate="one_to_one")
    mdl[vals] = mdl[vals].fillna(0).astype("int64")
    mdl["net_flow"] = mdl["return_count"] - mdl["rent_count"]
    mdl = (mdl.merge(cal_feat, on="date", how="left", validate="many_to_one")
              .merge(dim_hour, on="hour", how="left", validate="many_to_one")
              .merge(static, on="station_id", how="left", validate="many_to_one")
              .merge(prev[prev["ym"] == ym], on=["station_id", "ym", "hour"], how="left", validate="many_to_one"))
    assert len(mdl) == n0, "결합 후 행 수가 바뀌었습니다 (잘못된 병합)"
    for c in vals:
        assert int(mdl[c].sum()) == int(b[c].sum()), f"{ym}: {c} 합계 불일치"
    mdl["station_id"] = mdl["station_id"].astype("string")
    mdl = mdl.sort_values(["station_id", "date", "hour"]).reset_index(drop=True)
    save_parquet(mdl, MODEL_DIR / f"model_station_day_hour_{ym}.parquet")
    return len(mdl), int(mdl["rent_count"].sum()), float((mdl["rent_count"] == 0).mean())


rows, t0 = [], time.time()
for i, ym in enumerate(PERIOD_MONTHS, 1):
    n, s_, z = build_model_month(ym)
    rows.append({"ym": ym, "rows": n, "rent_total": s_, "zero_share": round(z, 3)})
    if i % 6 == 0 or i == N_MONTHS:
        print(f"  {i}/{N_MONTHS} ({ym}, {time.time() - t0:,.0f}초)")
model_log = pd.DataFrame(rows)
model_log.to_csv(LOG_DIR / "model_build_log.csv", index=False, encoding="utf-8-sig")
display(model_log)

## 8-2. 모델링 데이터 검증 (키 중복 · 기간 · 전월 정렬 · 합계)
**왜:** 잘못된 병합(행 증가)과 미래 정보 누수(같은 달 지하철 값 사용)를 막습니다.

In [ ]:
first_ym, last_ym, mid_ym = PERIOD_MONTHS[0], PERIOD_MONTHS[-1], PERIOD_MONTHS[len(PERIOD_MONTHS) // 2]
checks = []
for ym in sorted({first_ym, mid_ym, last_ym}):
    mdl = pd.read_parquet(MODEL_DIR / f"model_station_day_hour_{ym}.parquet")
    ok_align = None
    cand = mdl.dropna(subset=["prev_month_nearest_on_pd"])
    if len(cand):
        r = cand.iloc[len(cand) // 2]
        pm = (pd.Timestamp(r["date"]).to_period("M") - 1).strftime("%Y%m")
        src = feature_station_subway_month_hour[(feature_station_subway_month_hour["station_id"] == r["station_id"])
                                                & (feature_station_subway_month_hour["ym"] == pm)
                                                & (feature_station_subway_month_hour["hour"] == r["hour"])]
        ok_align = bool(len(src) == 1 and np.isclose(src["nearest_on_pd"].iloc[0], r["prev_month_nearest_on_pd"]))
    checks.append({"ym": ym, "rows": len(mdl), "key_dup": int(mdl.duplicated(["station_id", "date", "hour"]).sum()),
                   "in_range": bool(mdl["date"].between(START_TS, END_TS).all() and mdl["hour"].between(0, 23).all()),
                   "prev_month_null_share": round(float(mdl["prev_month_nearest_on_pd"].isna().mean()), 3),
                   "prev_month_aligned": ok_align})
df_checks = pd.DataFrame(checks)
display(df_checks)
assert (df_checks["key_dup"] == 0).all() and df_checks["in_range"].all()
assert df_checks.loc[df_checks["ym"] == first_ym, "prev_month_null_share"].iloc[0] == 1.0, "첫 달 전월 변수는 모두 결측이어야 합니다."
assert df_checks["prev_month_aligned"].dropna().all(), "전월 변수가 한 달 전 값과 맞지 않습니다."
model_rent = int(model_log["rent_total"].sum())
mart_rent = int(df_A.loc[smatch(df_A["station_id"], r"^ST-"), "rent_count"].sum())
assert model_rent == mart_rent, f"모델 데이터 대여 합계 {model_rent} ≠ 집계표 {mart_rent}"
print(f"모델 데이터 대여 합계 = 집계표 대여 합계 = {model_rent:,} ✔")

## 8-3. 컬럼 사전 (`model_feature_dictionary.csv`)
**왜:** 모델링을 맡은 팀원이 컬럼 뜻과 역할(타깃/변수/입력 금지)을 바로 알 수 있게 합니다.

In [ ]:
feature_dict = pd.DataFrame([
    ("station_id", "대여소 ID", "키"), ("date", "날짜", "키"), ("hour", "시(0~23)", "키/변수"), ("ym", "연월", "키"),
    ("rent_count", "대여 이벤트 건수", "타깃(y)"),
    ("completed_rent_count", "정상 완료 이용 건수", "입력 금지"), ("return_count", "정상 반납 건수", "입력 금지"),
    ("net_flow", "순유입(반납−대여)", "입력 금지"),
    ("year", "연", "변수"), ("month", "월", "변수"), ("dow", "요일(1=월)", "변수"), ("is_weekend", "주말", "변수"),
    ("is_holiday", "공휴일", "변수"), ("day_type", "요일구분", "변수"),
    ("is_commute_morning", "출근시간(7~9)", "변수"), ("is_commute_evening", "퇴근시간(17~19)", "변수"),
    ("hour_group", "시간대 그룹", "변수"), ("gu", "자치구", "변수"), ("hold_num", "거치대 수", "변수"),
    ("sta_lat", "위도", "변수"), ("sta_long", "경도", "변수"), ("has_coord", "좌표 정상", "변수"),
    ("nearest_distance_m", "가장 가까운 역 거리(m)", "변수"), ("is_subway_near_300", "역세권(300m)", "변수"),
    ("n_subway_300", "300m 안 역 수", "변수"), ("n_subway_500", "500m 안 역 수", "변수"),
    ("prev_month_nearest_on_pd", "전월 가장 가까운 역 하루 평균 승차(같은 시)", "변수"),
    ("prev_month_nearest_off_pd", "전월 가장 가까운 역 하루 평균 하차(같은 시)", "변수"),
    ("prev_month_w300_on_pd", "전월 300m 안 역 합계 하루 평균 승차", "변수"),
    ("prev_month_w300_off_pd", "전월 300m 안 역 합계 하루 평균 하차", "변수"),
], columns=["column", "뜻", "역할"])
mdl = pd.read_parquet(MODEL_DIR / f"model_station_day_hour_{last_ym}.parquet")
print("사전에 없는 컬럼:", sorted(set(mdl.columns) - set(feature_dict["column"])) or "없음 ✔")
feature_dict["결측률(마지막 달)"] = feature_dict["column"].map(lambda c: round(float(mdl[c].isna().mean()), 3) if c in mdl.columns else None)
feature_dict.to_csv(OUT_DIR / "model_feature_dictionary.csv", index=False, encoding="utf-8-sig")
display(feature_dict)

# 9. 최종 확인 — Parquet → DataFrame
## 9-1. 만든 파일 목록
**왜:** 빠진 결과 파일이나 저장 중 끊긴 임시 파일(`.tmp`)이 없는지 확인합니다.

In [ ]:
print("남은 임시 파일:", [str(p) for p in OUT_DIR.rglob("*.tmp")] or "없음 ✔")
inv = []
for p in sorted(OUT_DIR.rglob("*.parquet")):
    rel = p.relative_to(OUT_DIR)
    if rel.parts[0].startswith("_"):
        continue
    meta = pq.ParquetFile(p)
    inv.append({"dataset": rel.parts[0] if len(rel.parts) > 1 else rel.stem, "files": 1,
                "rows": meta.metadata.num_rows, "MB": p.stat().st_size / 1e6})
df_inv = pd.DataFrame(inv).groupby("dataset", as_index=False).agg(files=("files", "sum"), rows=("rows", "sum"), MB=("MB", "sum"))
df_inv["MB"] = df_inv["MB"].round(1)
display(df_inv)

## 9-2. 주요 결과를 DataFrame으로 불러와 확인
**왜:** 시각화·모델링 팀원이 쓰는 표를 실제로 읽어 행·열과 앞부분을 확인합니다.

| 변수 | 파일 | 한 행 | 키 |
|---|---|---|---|
| `df_station_both` | `bike_station_both` | 대여소 | `rent_id` |
| `df_station_link` | `station_subway_link` | 대여소 × 500m 안 역 | `rent_id`, `phys_key` |
| `df_mart_month` | `mart_station_month_hour/` | 대여소 × 연월 × 요일구분 × 시간 | `station_id`, `ym`, `day_type`, `hour` |
| `df_subway_month` | `subway_ride_month_hour` | 물리역 × 연월 × 시간 | `phys_key`, `ym`, `hour` |
| `df_feature_station` | `feature_station_subway` | 대여소 | `station_id` |
| `df_feature_month` | `feature_station_subway_month_hour` | 대여소 × 연월 × 시간 | `station_id`, `ym`, `hour` |
| `df_subway_day_full` | `subway_ride_day_hour_full` | 물리역 × 날짜 × 시간 (6절 실측이 있을 때만) | `phys_key`, `ride_date`, `hour` |
| `df_model_last` | `model_station_day_hour/` 마지막 달 | 대여소 × 날짜 × 시간 | `station_id`, `date`, `hour` |

In [ ]:
df_station_both    = pd.read_parquet(OUT_DIR / "bike_station_both.parquet")
df_station_link    = pd.read_parquet(OUT_DIR / "station_subway_link.parquet")
df_mart_month      = read_parquet_dir(OUT_DIR / "mart_station_month_hour")
df_subway_month    = pd.read_parquet(OUT_DIR / "subway_ride_month_hour.parquet")
df_feature_station = pd.read_parquet(OUT_DIR / "feature_station_subway.parquet")
df_feature_month   = pd.read_parquet(OUT_DIR / "feature_station_subway_month_hour.parquet")
_full = OUT_DIR / "subway_ride_day_hour_full.parquet"
df_subway_day_full = pd.read_parquet(_full) if _full.exists() else pd.DataFrame()
df_model_last      = pd.read_parquet(MODEL_DIR / f"model_station_day_hour_{last_ym}.parquet")

for name in ["df_station_both", "df_station_link", "df_mart_month", "df_subway_month", "df_feature_station",
             "df_feature_month", "df_subway_day_full", "df_model_last"]:
    d = globals()[name]
    print(f"\n[{name}] {d.shape[0]:,}행 × {d.shape[1]}열")
    display(d.head(3))